# 63 · Pathway analysis of human against mouse proximal tubule: the full 48-method grid (supplement)

This is the complete method grid behind the supplementary figure. It is a verbatim copy of the 48-method version of
notebook 61 (commit 7035b80); only the title and the results folder differ. "Notebook 61" in the text below means
that version. Notebook 61 now compares the five methods used in the main text.

Pathway analysis of a positional species difference can be organised in three ways:

| Family | Order of operations | Example |
|---|---|---|
| **A · pathway-first** | a statistic per gene → a test per pathway → look at its genes | T_spatial with the joint test (the primary method) |
| **C · score-first** | an activity score per pathway and structure → a positional test of the score | mean-z or AUCell-like score in a specimen × bin split plot |
| **B · gene-first** | cluster genes into modules → annotate modules with pathways | k-means of species-difference curves, then ORA |

Within family A, the gene statistic ranges from whole-PT averages to S1/S2/S3 segments to the
continuous coordinate, and the set test ranges over five kinds:
- over-representation;
- competitive rank tests (GSEA, matched rank-AUC, CAMERA);
- the joint test;
- self-contained tests at specimen level;
- self-contained tests at structure level.

The grid of combinations, with the ones dropped and why, was fixed in `plan_expansion.md` (workstream
B, notebook 61) before anything new was computed. Each method is identified by a code such as
**A6·JNT**: statistic A6 (T_spatial) with the joint test.

**Every method gets the same comparison:**
- calls under the species split and under both specimen relabelings, with notebook 31's
  specificity rule;
- calls collapsed to programs (the post-test driver-gene rule);
- overlap and rank agreement with the primary method;
- what the method returns;
- runtime;
- for specific methods, external replication with the overlap-preserving null.

**Assumptions.**
- SCF13 is treated as an exact coordinate.
- Specimens, not structures, are the replicates: 2 control mice and 2 cortex sections from 1 human
  donor, all male. Results are descriptive.
- Expensive statistics saved by notebooks 31, 37, 45, 50 and 56 are loaded and checked against their
  recorded values.

## 1 · Setup

In [ ]:
import argparse
import glob
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
nb12 = results_root / 'pt_pathway_remodeling_scfates'
nb31 = results_root / 'pt_pathway_method_selection'
nb37 = results_root / 'pt_pathway_final' / 'scfates'
nb45 = results_root / 'pt_revision_method' / 'tables'
nb45_cache = results_root / 'pt_revision_method' / 'stage_cache'
nb50 = results_root / 'pt_primary_pathways' / 'tables'
nb50_cache = results_root / 'pt_primary_pathways' / 'stage_cache'
nb56 = results_root / 'pt_pathway_continuous' / '56_taxonomy'
nb57 = results_root / 'pt_pathway_continuous' / '57_beyond_steps' / 'scf13' / 'tables'
nb58_cache = results_root / 'pt_pathway_continuous' / '58_descriptors' / 'scf13' / 'stage_cache'
external = data_root / 'external'
output = results_root / 'pt_pathway_methods_full_grid'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
REQUIRED = [results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv',
            nb31 / 'conventional_deseq2_all_partitions.csv', nb31 / 'conventional_signed_gsea_all_partitions.csv',
            nb31 / 'strategy_hits_all_partitions.csv', nb12 / 'conventional_pt_ora.csv',
            nb12 / 'pathway_spatial_correlation_tests.csv', nb12 / 'gene_statistics.csv',
            nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv', nb37 / 'tables' / 'table_S1_strategy_specificity.csv',
            nb37 / 'tables' / 'table_S2_pathways.csv', nb37 / 'tables' / 'split_plot_summary.csv',
            nb45 / 'anchor_our_cohort.csv', nb50 / 'cohort_step_screen_per_pathway.csv',
            nb56 / 'tables' / 'f_taxonomy.csv', nb56 / 'tables' / 'f_species_calls_by_approach.csv',
            nb57 / 'per_pathway_beyond_steps.csv',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 12, 31, 37, 45, 50, 56, 57 and 58 first)')

NOTEBOOK_LOGIC_VERSION = '61.methods_compared.2'  # Bump when a cached calculation changes.
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05
LFC = 1.0                     # ORA: DE genes need |log2FC| >= 1 (notebook 12)
N_NULL, SEED = 9999, 12       # matched random sets (notebooks 12, 31, 37)
N_BINS, MIN_CELL = 8, 20      # split plot (notebook 37)
K_MODULES, KM_STARTS, KM_SEED, TOP_SHARE = 8, 20, 56, .10   # notebook 56
K_RANGE, N_RESAMPLE = range(3, 9), 20                       # module-count selection (notebook 56)
N_ROTATIONS, ROTATION_SEED = 999, 61                        # self-contained rotation tests
SSGSEA_ALPHA, SCU_DF1 = .25, 6                              # ssGSEA weight; T_spatial spline df
WGCNA_GENES, WGCNA_POWER, WGCNA_HEIGHT, WGCNA_MIN = 3000, 6, .9, 30
N_PERM_EXTERNAL, MIN_CALLS_EXTERNAL, MIN_CELLS_EXTERNAL = 999, 5, 50
EXTERNAL_SEED, EXTERNAL_CHUNKS = 50, 27         # 27 chunks of 37 permutations, each seeded by its own id
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '32'))  # parallel workers; never part of a cache key
EXAMPLE_BINS = 12             # specimen x coordinate-bin means in the worked examples (notebook 41)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import sparse
from joblib import Parallel, delayed, parallel_config
from scipy.stats import chi2, f as f_dist, hypergeom, norm, rankdata, spearmanr
from threadpoolctl import threadpool_limits
from statsmodels.stats.multitest import multipletests

MM = 1 / 25.4
HUMAN, MOUSE, NULL, LIGHT, INK, MUTED, ACCENT = '#D55E00', '#0072B2', '#9A9A9A', '#D9D9D9', '#222222', '#666666', '#009E73'
SEGMENT_COLORS = {'S1': '#CDBCE3', 'S2': '#8A68BA', 'S3': '#3F2368'}
SPECIMENS = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
SPECIMEN_MARKER = {'Ctrl1A2': 'o', 'Ctrl1A4': 's', 'HUK1_COR1': 'o', 'HUK1_MED1': 's'}
FAMILY_COLORS = {'A · pathway-first': '#2a78d6', 'C · score-first': '#eda100', 'B · gene-first': '#1baf7a'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)
print('Results folder:', output, '· parallel workers:', N_JOBS)

## 2 · Structures, genes, pathways, partitions and shared helpers

All methods use notebook 37's construction on SCF13:
- the 12,272 PT structures inside every specimen's 1st–99th percentile range of the coordinate;
- the 11,071 genes measured in both species and detected in ≥ 2% of structures;
- the 1,513 Reactome, Hallmark and KEGG pathways with 10–300 tested genes.

The **species split** puts the two human sections against the two mice. Each **relabeling** pairs one
human section with one mouse. In relabeled models the species effect is a nuisance term, so only
specimen differences remain to be found.

A method is **specific** when NCDR (mean relabeled calls ÷ species calls) is below 0.25 and each
relabeling calls fewer than 5% of what was tested.

In [ ]:
from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_calibration import balanced_partitions, contrast_designs, nested_partial_f, pathway_scores
from pseudospace.pathway_decoys import joint_matched_test
from pseudospace.pathway_remodeling import correlation_adjusted_rank_tests, matched_pathway_tests
from pseudospace.split_plot import log_cpm, moderated_f, squeeze_var
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots, make_gam_design

inputs = pt_inputs(results_root, data_root, read_coordinate(results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'))
position, human, specimen, segment = (inputs[k] for k in ('position', 'human', 'specimen', 'segment'))
genes, gene_sets, strata = inputs['genes'], inputs['gene_sets'], inputs['strata']
pathways = list(gene_sets)
partitions = balanced_partitions(specimen, np.where(human, 'human', 'mouse'))
SWAPS = [p for p in partitions if p != 'species']
PARTS = ['species', *SWAPS]
ALL_SPECIMENS = SPECIMENS['mouse'] + SPECIMENS['human']
assert len(position) == 12272 and len(genes) == 11071 and len(pathways) == 1513, 'notebook 37 universe changed'
table37 = pd.read_csv(nb37 / 'tables' / 'table_S2_pathways.csv').set_index('pathway_id')
NAMES = table37.name.reindex(pathways)
VIF = table37.vif.reindex(pathways)
spec37 = pd.read_csv(nb37 / 'tables' / 'table_S1_strategy_specificity.csv').set_index('strategy')
gene37 = pd.read_csv(nb37 / 'tables' / 'gene_level_statistics.csv', index_col=0).T_spatial.reindex(genes)
cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION])
input_key = {'Y': digest(inputs['Y']), 'position': position, 'specimen': specimen, 'genes': genes}
grid = inputs['grid']
knots = gam_internal_knots(position, basis_df=6)
member_matrix = None   # set below
print('Partitions:', partitions)

CALLS, EVIDENCE, DIRECTION, METHODS = {}, {}, {}, []
FAMILY = {'A': 'A · pathway-first', 'C': 'C · score-first', 'B': 'B · gene-first'}
RETURNS = ['yes/no', 'direction', 'location', 'shape', 'genes']


def register(key, label, statistic, test, status, calls, *, evidence=None, direction=None, tested=None,
             returns=('yes/no',), runtime=None, demo=False, note=''):
    """Record one method under the common comparison.

    ``calls``: partition -> called pathway ids (species and both relabelings where defined); ``evidence``: per-pathway
    |z| or -log10 p for the species split; ``status``: E (in notebook 61 before the expansion), U (upstream, loaded and
    checked) or N (new); ``runtime``: seconds of the uncached computation, or 'loaded'.
    """
    CALLS[key] = {p: set(calls[p]) for p in PARTS if p in calls}
    if evidence is not None:
        EVIDENCE[key] = pd.Series(evidence, dtype=float).reindex(pathways)
    if direction is not None:
        DIRECTION[key] = pd.Series(direction, dtype=float).reindex(pathways)
    relabels = [len(CALLS[key][s]) for s in SWAPS if s in CALLS[key]]
    n_species, n_tested = len(CALLS[key]['species']), tested or len(pathways)
    ncdr = np.mean(relabels) / n_species if relabels and n_species else (0. if relabels else np.nan)
    passes = (bool(n_species and ncdr < NCDR_MAX and max(relabels) < NULL_RATE * n_tested) if relabels else None)
    METHODS.append({'key': key, 'family': FAMILY[key[0]], 'method': label + (' [demonstration]' if demo else ''),
                    'statistic': statistic, 'set test': test, 'status': status, 'tested': n_tested,
                    'species calls': n_species, 'relabeled calls': ', '.join(map(str, relabels)) if relabels else 'n/a',
                    'NCDR': ncdr, 'specific': passes, 'runtime (s)': runtime,
                    **{r: r in returns for r in RETURNS}, 'note': note})
    print(f"{key:<14} {n_species:>5} species calls; relabeled {relabels or 'n/a'}; specific {passes}")


def timed(stage, compute, kind='frame', **cache_kwargs):
    """Cache a single (serial) computation together with its uncached wall time; returns (result, seconds)."""
    def wrapped():
        t0 = time.perf_counter()
        out = compute()
        seconds = time.perf_counter() - t0
        if kind == 'frame':
            return out.assign(__seconds=seconds)
        out = dict(out)
        out['__seconds'] = np.array(seconds)
        return out
    if kind == 'frame':
        result = cached_frame(stage, wrapped, root=cache, code=code, **cache_kwargs)
        return result.drop(columns='__seconds'), float(result['__seconds'].iloc[0])
    result = cached_payload(stage, wrapped, root=cache, code=code, **cache_kwargs)
    return {k: v for k, v in result.items() if k != '__seconds'}, float(result['__seconds'])


def _run_task(function, args):
    t0 = time.perf_counter()
    with threadpool_limits(1):
        out = function(*args)
    return out, time.perf_counter() - t0


def fan_out(function, task_args, n_jobs=None):
    """``function(*args)`` for every args tuple, in task order, on ``n_jobs`` loky workers.

    Every task runs with one BLAS thread, also when ``n_jobs`` is 1, so floating-point results do not
    depend on the number of workers. Random tasks carry their own seeds, derived from fixed task ids.
    Large array arguments are memory-mapped by joblib. Returns (results, seconds per task).
    """
    jobs = N_JOBS if n_jobs is None else n_jobs
    if jobs == 1 or len(task_args) == 1:
        pairs = [_run_task(function, a) for a in task_args]
    else:
        with parallel_config(backend='loky', inner_max_num_threads=1):
            pairs = Parallel(n_jobs=min(jobs, len(task_args)))(delayed(_run_task)(function, a) for a in task_args)
    return [p[0] for p in pairs], [p[1] for p in pairs]


def cached_tasks(stage, function, tasks, **cache_kwargs):
    """Fan out ``tasks`` (task id -> args tuple) and cache every result as one stage-cache entry.

    Each task returns a dict of arrays. The entry stores them as '<task id>|<key>' with the task's
    own compute time, so the recorded runtime is the serial-equivalent time whatever ``n_jobs`` is.
    The cache key is the stage, ``params``, ``inputs`` and the code version, never ``n_jobs``.
    Returns {task id: (result dict, seconds)}.
    """
    ids = list(tasks)

    def compute():
        results, seconds = fan_out(function, [tasks[i] for i in ids])
        payload = {}
        for i, result, sec in zip(ids, results, seconds):
            payload.update({f'{i}|{k}': np.asarray(v) for k, v in result.items()})
            payload[f'{i}|__seconds'] = np.array(sec)
        return payload
    stored = cached_payload(stage, compute, root=cache, code=code, **cache_kwargs)
    out = {i: {} for i in ids}
    for key, value in stored.items():
        i, k = key.rsplit('|', 1)
        out[i][k] = value
    return {i: (out[i], float(out[i].pop('__seconds'))) for i in ids}


def slug(p):
    return p.replace(':', '_').replace('+', '_')


def membership(sets, universe):
    index = pd.Index(universe)
    rows, cols = [], []
    for i, members in enumerate(sets.values()):
        idx = index.get_indexer(members)
        idx = idx[idx >= 0]
        rows.extend([i] * len(idx)), cols.extend(idx)
    return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(sets), len(index)))


member_matrix = membership(gene_sets, genes)
member_size = np.asarray(member_matrix.sum(axis=1)).ravel()


def ora_pvalues(selected, matrix):
    """One-sided hypergeometric over-representation p for a boolean gene selection."""
    selected = np.asarray(selected, bool)
    if not selected.any():
        return np.ones(matrix.shape[0])
    k = np.asarray(matrix @ selected.astype(float)).ravel()
    return hypergeom.sf(k - 1, len(selected), np.asarray(matrix.sum(axis=1)).ravel(), selected.sum())


def bh(p):
    return multipletests(np.nan_to_num(np.asarray(p, float), nan=1.), method='fdr_bh')[1]


def newest(folder, stage, kind='npz'):
    files = glob.glob(str(folder / f'{stage}__*.{kind}'))
    if not files:
        raise FileNotFoundError(f'{stage} not found in {folder}')
    files.sort(key=lambda p: json.loads(Path(p + '.key.json').read_text())['written_utc'])
    if kind == 'csv':
        return pd.read_csv(files[-1])
    with np.load(files[-1]) as stored:
        return {k: stored[k] for k in stored.files}


def check_counts(key, strategy):
    row = spec37.loc[strategy]
    assert [len(CALLS[key][p]) for p in PARTS] == [row.species_calls, row.relabel_calls_Ctrl1A2, row.relabel_calls_Ctrl1A4], \
        (key, [len(CALLS[key][p]) for p in PARTS])

## 3 · Family A · pathway-first: a statistic per gene, then a test per pathway

**Idea.** Compute one statistic per gene for the species difference, then ask of each pathway whether
its members stand out. The set test decides what "stand out" means:

- **Competitive** tests ask whether the members differ **more than other genes**. These are:
  over-representation (ORA), preranked GSEA, the matched rank-AUC, CAMERA and the joint test.
  - A competitive test ranks pathways against each other. It is robust to effects shared by every
    gene, such as a global flattening of human gradients or probe-panel offsets.
  - It is also sensitive to specimen effects that move whole pathways together. That is why the
    joint test divides by the variance inflation from inter-gene correlation.
- **Self-contained** tests ask whether **any member differs at all**, ignoring other genes. These
  are ROAST-like rotation tests and global tests.
  - At **specimen level** (pseudobulk, with specimens as replicates) the answer is honest but
    unselective: when most genes differ between species, most pathways do.
  - At **structure level** (12,000 structures treated as replicates) every pathway differs, because
    specimens are pseudoreplicated. One row demonstrates this.

**The gene statistics** run from coarse to fine. The table gives each statistic with its set-test
codes (E: in notebook 61 before the expansion; U: computed upstream and checked; N: new):

| Code | Statistic | Set tests run |
|---|---|---|
| A1 | whole-PT pseudobulk DESeq2 (average difference) | ORAc (E), GSEA (E), JNT (U), SCs (N) |
| A2 | whole-PT GAM offset `T_level` | MAUC (U), JNT (U) |
| A3 | S1/S2/S3 pseudobulk DESeq2 | ORAc (E), GSEA (E), JNT (U), SCs (N) |
| A0 | segment markers, segment against rest (a zonation question, not a species one) | ORAc (E) |
| A4 | coordinate-free S1/S2/S3 steps | JNT (E), MAUC (U), GSEA (N), SCs (N) |
| A5 | six coordinate bins | JNT (E), MAUC (U), GSEA (N) |
| A6 | **T_spatial**, smooth species × position | JNT (E, primary), MAUC (E), GSEA (E), CAM (U), ORAc (N), ORAr (N, demonstration), SCs (N), SCu (N, demonstration) |
| A7 | `T_total` (offset + shape; tradeSeq `patternTest` analogue) | MAUC (U), JNT (U) |
| A8 | per-species association (tradeSeq `associationTest`), lists compared | JNT per group (E) |
| A9 | start versus end (tradeSeq `startVsEndTest`) | JNT (E) |
| A10 | specimen-level split-plot gene F (8 bins) | MAUC (U), JNT (U), GSEA (N) |

Set-test codes: ORAc = ORA on a calibrated DE list; ORAr = ORA on the top 10% of a ranking;
MAUC = matched rank-AUC; JNT = MAUC ÷ √VIF; CAM = CAMERA; SCs/SCu = self-contained at specimen /
structure level. The dropped combinations and their reasons are in `plan_expansion.md`.

### 3.1 · Shared inputs for family A

Shared inputs:
- notebook 31's DESeq2 and GSEA;
- notebook 37's gene statistics, matched tests, split-plot results and residual correlations;
- notebook 45's cell summaries and pathway sets for our cohort;
- pseudobulks summed from the structure counts.

In [ ]:
from pseudospace.pathways import build_pathway_membership
from pseudospace.specimen_null import matched_auc_null, pseudobulk_t, summary_partial_f

deseq = pd.read_csv(nb31 / 'conventional_deseq2_all_partitions.csv')
gsea31 = pd.read_csv(nb31 / 'conventional_signed_gsea_all_partitions.csv')
hits31 = pd.read_csv(nb31 / 'strategy_hits_all_partitions.csv')
tests37 = pd.read_csv(nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv')
tests37 = tests37[tests37.partition.isin(PARTS)]
recomputed = joint_matched_test(tests37.drop(columns=['z_matched', 'vif', 'z_joint', 'p_joint', 'q_joint']), VIF)
np.testing.assert_allclose(recomputed.z_joint, tests37.z_joint, rtol=1e-9)   # Guard: joint z = matched z ÷ √VIF
gene_scores37 = {p: newest(nb37 / 'stage_cache', f'gene_scores_{slug(p)}') for p in PARTS}
split37 = {p: newest(nb37 / 'stage_cache', f'split_plot_{slug(p)}') for p in PARTS}
np.testing.assert_allclose(gene_scores37['species']['T_spatial'], gene37.to_numpy(), rtol=1e-6)   # Guard
pool = newest(nb45_cache, 'pool_our_cohort')
assert list(pool['genes'].astype(str)) == list(genes), 'notebook 45 pool genes differ from the universe'
universe12 = pd.read_csv(nb12 / 'gene_statistics.csv', index_col=0).index.tolist()
coverage45 = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), universe12,
    ortholog_map=pd.read_csv(results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'), library_name=lib,
    tested=universe12, min_genes=1) for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')], ignore_index=True)
coverage45['pathway_id'] = coverage45.library + '::' + coverage45.pathway
coverage45 = coverage45[coverage45.n_tested.between(10, min(300, len(universe12) - 1))]
sets45 = {row.pathway_id: list(row.genes_present) for row in coverage45.itertuples()}
pool_genes = set(pool['genes'].astype(str))
sets45 = {p: [g for g in sets45[p] if g in pool_genes] for p in pool['pathways'].astype(str)}   # notebook 45's order and members
pool_vif = pd.Series(pool['vif'], index=list(sets45))
print('Notebook 45 sets differ from notebook 37 sets in', sum(set(sets45[p]) != set(gene_sets[p]) for p in pathways), 'pathway')
LOADED = 'loaded'


def pseudobulk(labels):
    """Raw-count sums and library sizes per label (rows of the returned matrix follow sorted labels)."""
    names, codes = np.unique(labels, return_inverse=True)
    indicator = sparse.csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))), shape=(len(names), len(codes)))
    return names, np.asarray((indicator @ inputs['counts']).todense()), np.asarray(indicator @ inputs['library']).ravel()


def rotation_test(y, design, tested, sets_matrix=member_matrix, n_rot=N_ROTATIONS, seed=ROTATION_SEED):
    """ROAST-like self-contained test: does any member change? Set statistic = members' mean moderated F.

    OLS on pseudobulk log2-CPM; the moderation prior is estimated once from the observed residual variances
    and held fixed; random orthogonal rotations of the tested-plus-residual effects are shared by all genes
    (keeping inter-gene correlation) and all sets. Simplification of limma's ROAST: no precision weights.
    """
    y, x = np.asarray(y, float), np.asarray(design, float)
    n, p = x.shape
    tested = list(tested)
    nuisance = [c for c in range(p) if c not in tested]
    if np.linalg.matrix_rank(x) < p or n - p < 1:
        raise ValueError('Design must be full rank with residual degrees of freedom.')
    q_full, _ = np.linalg.qr(x[:, nuisance + tested], mode='complete')
    q, d = len(tested), n - p
    block = (q_full.T @ y)[len(nuisance):]
    s2 = (block[q:] ** 2).sum(axis=0) / d
    floor = np.median(s2[s2 > 0])
    _, d0, s02 = squeeze_var(np.where(s2 > 0, s2, floor), d)

    def mod_f(b):
        s2b = (b[q:] ** 2).sum(axis=0) / d
        post = (d0 * s02 + d * s2b) / (d0 + d) if np.isfinite(d0) else np.full_like(s2b, s02)
        return (b[:q] ** 2).sum(axis=0) / q / np.maximum(post, 1e-12)

    size = np.asarray(sets_matrix.sum(axis=1)).ravel()
    observed = (sets_matrix @ mod_f(block)) / size
    rng = np.random.default_rng(seed)
    exceed = np.zeros(len(observed))
    for _ in range(n_rot):
        rot, r = np.linalg.qr(rng.normal(size=(q + d, q + d)))
        rot = rot * np.sign(np.diag(r))
        exceed += (sets_matrix @ mod_f(rot @ block)) / size >= observed - 1e-12
    return (1 + exceed) / (n_rot + 1)


def gsea_task(statistic, gene_names, sets, weight):
    """Preranked GSEA (gseapy multilevel, one thread, seed 12); NES and nominal p aligned to ``sets``."""
    import gseapy as gp
    ranking = (pd.Series(statistic, index=gene_names).rename_axis('gene').reset_index(name='stat')
               .sort_values(['stat', 'gene'], ascending=[False, True], kind='stable'))
    table = gp.prerank(rnk=ranking, gene_sets=sets, min_size=10, max_size=300, weight=weight, method='multilevel',
                       seed=SEED, threads=1, outdir=None, no_plot=True, verbose=False).res2d.set_index('Term')
    return {'NES': pd.to_numeric(table.NES).reindex(list(sets)).to_numpy(float),
            'p': pd.to_numeric(table['NOM p-val']).reindex(list(sets)).to_numpy(float)}


def gsea_method(label_stage, source):
    """Preranked GSEA (weight 0) of an unsigned statistic, one task per partition; call = NES > 0 and BH q <= 0.05."""
    runs = cached_tasks(f'gsea_{label_stage}', gsea_task, {p: (source(p), genes, gene_sets, 0) for p in PARTS},
                        params={'weight': 0, 'seed': SEED}, inputs={'stat': [source(p) for p in PARTS], 'sets': gene_sets})
    calls = {}
    for p, (r, _) in runs.items():
        q = bh(r['p'])
        calls[p] = np.array(pathways)[(r['NES'] > 0) & (q <= ALPHA)]
    return calls, -np.log10(pd.Series(runs['species'][0]['p'], index=pathways).clip(lower=1e-300)), sum(sec for _, sec in runs.values())


def rotation_method(stage, designs, y):
    """Rotation tests, one task per partition; ``designs``: partition -> (design, tested columns)."""
    runs = cached_tasks(stage, lambda y_, x, t: {'p': rotation_test(y_, x, t)},
                        {p: (y, designs[p][0], designs[p][1]) for p in PARTS},
                        params={'n_rot': N_ROTATIONS, 'seed': ROTATION_SEED},
                        inputs={'y': digest(y), 'designs': [digest(designs[p][0]) for p in PARTS]})
    return {p: r['p'] for p, (r, _) in runs.items()}, sum(sec for _, sec in runs.values())

### 3.2 · A1–A3 and A0: whole-PT and S1/S2/S3 pseudobulks

DESeq2 statistics are on integer counts summed over each specimen (and segment). The species design
is `~species`; a relabeled design is `~species + group`, so it has one fewer residual degree of
freedom.

In [ ]:


def deseq_ora(frame, scopes):
    columns, labels = [], []
    for scope in scopes:
        rows = frame[frame.scope.eq(scope)].dropna(subset=['stat']).set_index('gene')
        matrix = membership(gene_sets, list(rows.index))
        for direction in (1, -1):
            selected = rows.padj.le(ALPHA).to_numpy() & (direction * rows.log2FoldChange).ge(LFC).to_numpy()
            columns.append(ora_pvalues(selected, matrix))
            labels.append((scope, direction))
    return pd.DataFrame(np.column_stack(columns), index=pathways, columns=pd.MultiIndex.from_tuples(labels))


def ora_summary(pvalues):
    q = pd.DataFrame(bh(pvalues.to_numpy().ravel()).reshape(pvalues.shape), index=pvalues.index, columns=pvalues.columns)
    best = pvalues.idxmin(axis=1)
    return q.le(ALPHA).any(axis=1), -np.log10(pvalues.min(axis=1).clip(lower=1e-300)), best.map(lambda c: c[1])


t0 = time.perf_counter()
ora = {(family, p): ora_summary(deseq_ora(deseq[deseq.partition.eq(p)], scopes))
       for family, scopes in (('whole', ['whole_PT']), ('segments', ['PT-S1', 'PT-S2', 'PT-S3'])) for p in PARTS}
ora_seconds = (time.perf_counter() - t0) / 2
saved56 = pd.read_csv(nb56 / 'tables' / 'f_species_calls_by_approach.csv', index_col=0)
for key, family, label, stat, col56 in (('A1·ORAc', 'whole', 'Whole-PT DE + ORA', 'whole-PT DESeq2', 'Whole-PT pseudobulk DE + ORA'),
                                        ('A3·ORAc', 'segments', 'S1/S2/S3 DE + ORA', 'S1/S2/S3 DESeq2', 'S1/S2/S3 pseudobulk DE + ORA')):
    register(key, label, stat, 'ORA on DE lists', 'E', {p: [x for x, h in ora[(family, p)][0].items() if h] for p in PARTS},
             evidence=ora[(family, 'species')][1], direction=ora[(family, 'species')][2], runtime=ora_seconds,
             returns=('yes/no', 'direction', 'genes') + (('location',) if family == 'segments' else ()))
    assert CALLS[key]['species'] == set(saved56.index[saved56[col56]])   # Guard: notebook 56

whole = gsea31[gsea31.family.eq('whole_PT')]
register('A1·GSEA', 'Whole-PT DE + signed GSEA', 'whole-PT DESeq2', 'preranked GSEA', 'E',
         {p: whole.pathway_id[whole.partition.eq(p) & whole.q.le(ALPHA)] for p in PARTS},
         evidence=-np.log10(whole[whole.partition.eq('species')].set_index('pathway_id').p.clip(lower=1e-300)),
         direction=np.sign(whole[whole.partition.eq('species')].set_index('pathway_id').NES),
         tested=whole.pathway_id.nunique(), runtime=LOADED, returns=('yes/no', 'direction', 'genes'))
check_counts('A1·GSEA', '1 · Whole-PT pseudobulk + signed GSEA')
segs = gsea31[gsea31.family.eq('segments')]
best_segment = segs[segs.partition.eq('species')].sort_values('p').drop_duplicates('pathway_id').set_index('pathway_id')
register('A3·GSEA', 'S1/S2/S3 DE + signed GSEA', 'S1/S2/S3 DESeq2', 'preranked GSEA per segment', 'E',
         {p: segs.pathway_id[segs.partition.eq(p) & segs.q.le(ALPHA)].unique() for p in PARTS},
         evidence=-np.log10(best_segment.p.clip(lower=1e-300)), direction=np.sign(best_segment.NES),
         tested=segs.pathway_id.nunique(), runtime=LOADED, returns=('yes/no', 'direction', 'location', 'genes'))
check_counts('A3·GSEA', '2 · S1/S2/S3 pseudobulk + signed GSEA')

# Notebook 45's pseudobulk joint rank tests (two-sided matched rank-AUC of the moderated t, divided by √VIF).
anchor = pd.read_csv(nb45 / 'anchor_our_cohort.csv').set_index('strategy')
pool_donors = list(pool['donors'].astype(str))
species_of = {d: float(d in SPECIMENS['human']) for d in pool_donors}
designs45 = {'species': (species_of, None)}
designs45.update({s: ({d: float(d in partitions[s]) for d in pool_donors}, species_of) for s in SWAPS})


def pool_rank_tests():
    out = {}
    for p, (group, nuisance) in designs45.items():
        cols = [np.ones(len(pool_donors)), [group[d] for d in pool_donors]] + ([[nuisance[d] for d in pool_donors]] if nuisance else [])
        design = np.column_stack(cols).astype(float)
        t = {'whole': pseudobulk_t(pool['counts'].sum(axis=1), pool['library'].sum(axis=1), design, 1)[0]}
        for k, seg in enumerate(pool['segments'].astype(str)):
            t[seg] = pseudobulk_t(pool['counts'][:, k], pool['library'][:, k], design, 1)[0]
        tests = matched_auc_null(pd.DataFrame(t, index=genes), sets45, pool['strata'])
        for stat, block in tests.groupby('statistic'):
            out[f'{p}|{stat}'] = (block.set_index('pathway_id').z_matched / np.sqrt(pool_vif)).reindex(list(sets45)).to_numpy()
    return out


rank45, sec45 = timed('pool_rank_tests', pool_rank_tests, kind='payload', inputs={'pool': digest(pool['counts'])})
for key, label, stat, scopes, anchor_row in (
        ('A1·JNT', 'Whole-PT pseudobulk, joint rank test', 'whole-PT moderated t', ['whole'], 'whole-PT pseudobulk, joint rank test'),
        ('A3·JNT', 'S1/S2/S3 pseudobulk, joint rank test', 'S1/S2/S3 moderated t', ['S1', 'S2', 'S3'], 'S1/S2/S3 pseudobulk, joint rank test')):
    calls, evid = {}, None
    for p in PARTS:
        pz = np.column_stack([2 * norm.sf(np.abs(rank45[f'{p}|{s}'])) for s in scopes])
        q = bh(pz.ravel()).reshape(pz.shape)
        calls[p] = np.array(list(sets45))[(q <= ALPHA).any(axis=1)]
        if p == 'species':
            evid = pd.Series(np.max(np.column_stack([np.abs(rank45[f'species|{s}']) for s in scopes]), axis=1), index=list(sets45))
    register(key, label, stat, 'two-sided joint rank test', 'U', calls, evidence=evid, runtime=sec45,
             returns=('yes/no', 'genes') + (('location',) if len(scopes) > 1 else ()))
    row = anchor.loc[anchor_row]
    assert [len(CALLS[key][p]) for p in PARTS] == [row.species, row.relabel_Ctrl1A2, row.relabel_Ctrl1A4], key   # Guard: notebook 45

# Specimen-level self-contained rotation tests on whole-PT and specimen x segment pseudobulks.
whole_names, whole_counts, whole_library = pseudobulk(specimen)
seg_names, seg_counts, seg_library = pseudobulk(np.char.add(np.char.add(specimen.astype(str), '|'), segment.astype(str)))
seg_spec = np.array([n.split('|')[0] for n in seg_names])
seg_seg = np.array([n.split('|')[1] for n in seg_names])


def pseudobulk_designs(kind):
    """Designs and tested columns per partition for the whole-PT, segment and step rotation tests."""
    out = {}
    for p in PARTS:
        if kind == 'whole':
            sp = np.isin(whole_names, SPECIMENS['human']).astype(float)
            g = np.isin(whole_names, partitions[p]).astype(float)
            out[p] = ((np.column_stack([np.ones(4), sp]), [1]) if p == 'species'
                      else (np.column_stack([np.ones(4), sp, g]), [2]))
            continue
        sp = np.isin(seg_spec, SPECIMENS['human']).astype(float)
        g = np.isin(seg_spec, partitions[p]).astype(float)
        s2, s3 = (seg_seg == 'PT-S2').astype(float), (seg_seg == 'PT-S3').astype(float)
        spec_cols = np.column_stack([(seg_spec == n).astype(float) for n in ALL_SPECIMENS])
        if kind == 'segments':      # A3: any species difference within segments
            design = np.column_stack([np.ones(len(sp)), s2, s3, sp, sp * s2, sp * s3]
                                     + ([] if p == 'species' else [g, g * s2, g * s3]))
            out[p] = (design, [3, 4, 5] if p == 'species' else [6, 7, 8])
        else:                       # A4: species x segment beyond specimen offsets
            design = np.column_stack([spec_cols, s2, s3, sp * s2, sp * s3] + ([] if p == 'species' else [g * s2, g * s3]))
            out[p] = (design, [6, 7] if p == 'species' else [8, 9])
    return out


whole_y, seg_y = log_cpm(whole_counts, whole_library), log_cpm(seg_counts, seg_library)
for key, label, stat, kind in (('A1·SCs', 'Whole-PT pseudobulk, self-contained rotation test', 'whole-PT log-CPM', 'whole'),
                               ('A3·SCs', 'S1/S2/S3 pseudobulk, self-contained rotation test', 'S1/S2/S3 log-CPM', 'segments')):
    pv, sec = rotation_method(f'rotation_{kind}', pseudobulk_designs(kind), whole_y if kind == 'whole' else seg_y)
    register(key, label, stat, 'self-contained, specimen level', 'N',
             {p: np.array(pathways)[bh(pv[p]) <= ALPHA] for p in PARTS},
             evidence=-np.log10(pd.Series(pv['species'], index=pathways)), runtime=sec, returns=('yes/no', 'genes'))

markers = pd.read_csv(nb12 / 'conventional_pt_ora.csv')
markers = markers[markers.scope.eq('segment_vs_rest')].copy()
np.testing.assert_allclose(bh(markers.p_value), markers.q_family, rtol=1e-9)   # Guard
best_marker = markers.sort_values('p_value').drop_duplicates('pathway_id').set_index('pathway_id')
register('A0·markers', 'Segment markers + ORA (species pooled; zonation, not species)', 'segment vs rest DESeq2', 'ORA', 'E',
         {'species': markers.pathway_id[markers.q_family.le(ALPHA)].unique()},
         evidence=-np.log10(best_marker.p_value.clip(lower=1e-300)), tested=markers.pathway_id.nunique(), runtime=LOADED,
         returns=('yes/no', 'location', 'genes'))

### 3.3 · A2 and A7: structure-level offset and offset-plus-shape statistics

`T_level` (a species offset) and `T_total` (offset or shape, the analogue of tradeSeq's `patternTest`)
come from notebook 37's nested GAMs. Both fail the relabeling rule (notebook 37); they are included so
that every statistic is on the same footing.

In [ ]:


def from37(statistic, column):
    block = tests37[tests37.statistic.eq(statistic)]
    return ({p: block.pathway_id[block.partition.eq(p) & block.effect.gt(0) & block[column].le(ALPHA)] for p in PARTS},
            block[block.partition.eq('species')].set_index('pathway_id'))


for key, stat, column, test, strategy, returns in (
        ('A2·MAUC', 'T_level', 'q_normal', 'matched rank-AUC', '4 · Gene GAM offset (T_level)', ('yes/no',)),
        ('A2·JNT', 'T_level', 'q_joint', 'joint test', '4 · Gene GAM offset (T_level) (joint test)', ('yes/no',)),
        ('A7·MAUC', 'T_total', 'q_normal', 'matched rank-AUC', '5 · Gene GAM total (T_total)', ('yes/no',)),
        ('A7·JNT', 'T_total', 'q_joint', 'joint test', '5 · Gene GAM total (T_total) (joint test)', ('yes/no',))):
    calls, rows = from37(stat, column)
    register(key, f'GAM {stat}, {test}', f'structure-level {stat}', test, 'U', calls,
             evidence=rows.z_joint if column == 'q_joint' else rows.z_matched, runtime=LOADED, returns=returns)
    check_counts(key, strategy)

### 3.4 · A4 and A5: coordinate-free segment steps and six coordinate bins

A4 needs no coordinate: it is the species × S1/S2/S3 step model beyond a species offset, with segment
intercepts as the shared shape, computed exactly from notebook 45's cell summaries. A5 replaces the
three segments with six equal bins of the coordinate: a "fine discretisation" bridge to the
continuous methods.

In [ ]:
steps50 = pd.read_csv(nb50 / 'cohort_step_screen_per_pathway.csv').set_index('pathway_id')


def step_statistics():
    out = {}
    for p, (group, nuisance) in designs45.items():
        out[p] = summary_partial_f(pool, pool_donors, group, nuisance)['T_steps']
    return out


t_steps, sec_steps = timed('t_steps', step_statistics, kind='payload', inputs={'pool': digest(pool['mean'])})
step_tests = {p: matched_auc_null(pd.DataFrame({'T_steps': t_steps[p]}, index=genes), sets45, pool['strata']).set_index('pathway_id')
              for p in PARTS}
for p in PARTS:
    step_tests[p]['z_joint'] = step_tests[p].z_matched / np.sqrt(pool_vif)
register('A4·JNT', 'S1/S2/S3 steps, coordinate-free, joint test', 'T_steps (no coordinate)', 'joint test', 'E',
         {p: t.index[t.effect.gt(0) & (bh(norm.sf(t.z_joint)) <= ALPHA)] for p, t in step_tests.items()},
         evidence=step_tests['species'].z_joint, runtime=sec_steps, returns=('yes/no', 'location'))
assert CALLS['A4·JNT']['species'] == set(steps50.index[steps50.called.astype(bool)])          # Guard: notebook 50
assert [len(CALLS['A4·JNT'][s]) for s in SWAPS] == [0, 0]
register('A4·MAUC', 'S1/S2/S3 steps, matched rank-AUC', 'T_steps (no coordinate)', 'matched rank-AUC', 'U',
         {p: t.index[t.effect.gt(0) & (bh(norm.sf(t.z_matched)) <= ALPHA)] for p, t in step_tests.items()},
         evidence=step_tests['species'].z_matched, runtime=sec_steps, returns=('yes/no', 'location'))
row = anchor.loc['T_steps matched']
assert [len(CALLS['A4·MAUC'][p]) for p in PARTS] == [row.species, row.relabel_Ctrl1A2, row.relabel_Ctrl1A4]   # Guard
calls, evid, sec = gsea_method('t_steps', lambda p: t_steps[p])
register('A4·GSEA', 'S1/S2/S3 steps, preranked GSEA', 'T_steps (no coordinate)', 'preranked GSEA', 'N', calls,
         evidence=evid, runtime=sec + sec_steps, returns=('yes/no', 'location', 'genes'))
pv, sec = rotation_method('rotation_steps', pseudobulk_designs('steps'), seg_y)
register('A4·SCs', 'S1/S2/S3 steps, self-contained rotation test', 'specimen × segment log-CPM', 'self-contained, specimen level', 'N',
         {p: np.array(pathways)[bh(pv[p]) <= ALPHA] for p in PARTS}, evidence=-np.log10(pd.Series(pv['species'], index=pathways)),
         runtime=sec, returns=('yes/no', 'location', 'genes'))

calls, rows = from37('T_bins_spatial', 'q_joint')
register('A5·JNT', 'Six coordinate bins, joint test', 'species × 6 bins', 'joint test', 'E', calls, evidence=rows.z_joint,
         runtime=LOADED, returns=('yes/no', 'location'))
check_counts('A5·JNT', '7 · Gene GAM × 6 equal bins (joint test)')
calls, rows = from37('T_bins_spatial', 'q_normal')
register('A5·MAUC', 'Six coordinate bins, matched rank-AUC', 'species × 6 bins', 'matched rank-AUC', 'U', calls,
         evidence=rows.z_matched, runtime=LOADED, returns=('yes/no', 'location'))
check_counts('A5·MAUC', '7 · Gene GAM × 6 equal bins')
calls, evid, sec = gsea_method('t_bins', lambda p: gene_scores37[p]['T_bins_spatial'])
register('A5·GSEA', 'Six coordinate bins, preranked GSEA', 'species × 6 bins', 'preranked GSEA', 'N', calls, evidence=evid,
         runtime=sec, returns=('yes/no', 'location', 'genes'))

### 3.5 · A6: T_spatial along the coordinate, with every set test

T_spatial is the partial F of species × a 6-df spline beyond a species offset. It ranks genes but is
not itself a calibrated gene test, because its degrees of freedom count structures.
- **Calibrated gene list (ORAc).** The specimen-level split-plot F (notebook 37: species × position
  judged against specimen × position) supplies it.
- **Top 10% by T_spatial (ORAr).** The common practice, shown as a demonstration.
- **SCs** is the rotation test on the 8-bin split-plot pseudobulk.
- **SCu** treats the structures as independent, also a demonstration.

In [ ]:
calls, rows = from37('T_spatial', 'q_joint')
register('A6·JNT', 'T_spatial, joint test (primary)', 'T_spatial', 'joint test', 'E', calls, evidence=rows.z_joint,
         runtime=LOADED, returns=('yes/no', 'direction', 'location', 'shape', 'genes'))
check_counts('A6·JNT', '8 · Gene GAM × smooth position (T_spatial) (joint test)')
calls, rows = from37('T_spatial', 'q_normal')
register('A6·MAUC', 'T_spatial, matched rank-AUC', 'T_spatial', 'matched rank-AUC', 'E', calls, evidence=rows.z_matched,
         runtime=LOADED, returns=('yes/no', 'direction', 'location', 'shape', 'genes'))
check_counts('A6·MAUC', '8 · Gene GAM × smooth position (T_spatial)')
c3 = hits31[hits31.strategy.eq('8c · T_spatial, preranked GSEA')]
spatial_gsea = newest(nb37 / 'stage_cache', 'spatial_gsea_species', kind='csv').set_index('pathway_id')
register('A6·GSEA', 'T_spatial, preranked GSEA', 'T_spatial', 'preranked GSEA', 'E',
         {p: c3.pathway_id[c3.partition.eq(p) & c3.hit] for p in PARTS},
         evidence=-np.log10(spatial_gsea.p.clip(lower=1e-300)), runtime=LOADED, returns=('yes/no', 'direction', 'location', 'shape', 'genes'))
check_counts('A6·GSEA', '8c · T_spatial, preranked GSEA')

correlations37 = newest(nb37 / 'stage_cache', 'residual_correlations', kind='csv')
correlations37 = correlations37[correlations37.family.eq('all')].drop(columns='family')


def camera_task(statistic, gene_names, sets, correlations):
    frame = correlation_adjusted_rank_tests(pd.Series(statistic, index=gene_names), sets, correlations)
    frame = frame.set_index('pathway_id').reindex(list(sets))
    return {'p': frame.p_corr.to_numpy(float), 'auc': frame.auc.to_numpy(float)}


camera_runs = cached_tasks('camera', camera_task,
                           {p: (gene_scores37[p]['T_spatial'], genes, gene_sets, correlations37) for p in PARTS},
                           inputs={'scores': [digest(gene_scores37[p]['T_spatial']) for p in PARTS],
                                   'correlations': digest(correlations37)})
cam = {p: r['p'] for p, (r, _) in camera_runs.items()}
cam.update({p + '|auc': r['auc'] for p, (r, _) in camera_runs.items()})
sec = sum(sec for _, sec in camera_runs.values())
saved12 = pd.read_csv(nb12 / 'pathway_spatial_correlation_tests.csv').set_index('pathway_id').reindex(pathways)
np.testing.assert_allclose(cam['species'], saved12.p_corr, rtol=1e-6)   # Guard: notebook 12's CAMERA test
register('A6·CAM', 'T_spatial, CAMERA (unmatched, VIF)', 'T_spatial', 'CAMERA', 'U',
         {p: np.array(pathways)[(bh(cam[p]) <= ALPHA) & (cam[p + '|auc'] > .5)] for p in PARTS},
         evidence=-np.log10(pd.Series(cam['species'], index=pathways).clip(lower=1e-300)), runtime=sec, returns=('yes/no', 'direction', 'location', 'shape', 'genes'))

t0 = time.perf_counter()
ora_c, ora_r = {}, {}
for p in PARTS:
    significant = bh(split37[p]['p']) <= ALPHA
    pc = ora_pvalues(significant, member_matrix)
    ora_c[p] = (np.array(pathways)[bh(pc) <= ALPHA], pc, int(significant.sum()))
    top = gene_scores37[p]['T_spatial'] >= np.quantile(gene_scores37[p]['T_spatial'], 1 - TOP_SHARE)
    pr = ora_pvalues(top, member_matrix)
    ora_r[p] = (np.array(pathways)[bh(pr) <= ALPHA], pr)
sec_ora = time.perf_counter() - t0
print('Split-plot genes at BH ≤ 0.05 (the calibrated list):', {p: v[2] for p, v in ora_c.items()})
register('A6·ORAc', 'Split-plot significant genes + ORA', 'specimen-level split-plot F (BH ≤ 0.05 genes)', 'ORA', 'N',
         {p: v[0] for p, v in ora_c.items()}, evidence=-np.log10(pd.Series(ora_c['species'][1], index=pathways)),
         runtime=sec_ora / 2, returns=('yes/no', 'direction', 'location', 'shape', 'genes'))
register('A6·ORAr', 'Top 10% T_spatial + ORA', 'T_spatial (top decile)', 'ORA on a rank threshold', 'N',
         {p: v[0] for p, v in ora_r.items()}, evidence=-np.log10(pd.Series(ora_r['species'][1], index=pathways)),
         runtime=sec_ora / 2, demo=True, returns=('yes/no', 'direction', 'location', 'shape', 'genes'))

pb8 = newest(nb37 / 'stage_cache', 'pseudobulk_cells', kind='csv')
pb8_counts = pb8[[f'count_{g}' for g in genes]].to_numpy(float)
pb8_library = pb8['count___library__'].to_numpy(float)
pb8_spec = pb8.specimen.astype(str).to_numpy()
pb8_basis = make_gam_design(pb8.pseudospace_bin_centre.to_numpy(float), np.array([]))[:, 1:]


def split_plot_designs():
    spec_cols = np.column_stack([(pb8_spec == n).astype(float) for n in ALL_SPECIMENS])
    hum = np.isin(pb8_spec, SPECIMENS['human']).astype(float)[:, None]
    out = {}
    for p in PARTS:
        if p == 'species':
            design = np.column_stack([spec_cols, pb8_basis, hum * pb8_basis])
        else:
            g = np.isin(pb8_spec, partitions[p]).astype(float)[:, None]
            design = np.column_stack([spec_cols, pb8_basis, hum * pb8_basis, g * pb8_basis])
        out[p] = (design, list(range(design.shape[1] - 3, design.shape[1])))
    return out


pv, sec = rotation_method('rotation_split_plot', split_plot_designs(), log_cpm(pb8_counts, pb8_library))
register('A6·SCs', 'Split-plot pseudobulk, self-contained rotation test', 'specimen × 8-bin log-CPM', 'self-contained, specimen level', 'N',
         {p: np.array(pathways)[bh(pv[p]) <= ALPHA] for p in PARTS}, evidence=-np.log10(pd.Series(pv['species'], index=pathways)),
         runtime=sec, returns=('yes/no', 'direction', 'location', 'shape', 'genes'))

t0 = time.perf_counter()
scu = {}
for p in PARTS:
    total = np.asarray(member_matrix @ gene_scores37[p]['T_spatial']).ravel() * SCU_DF1
    scu[p] = chi2.sf(total, SCU_DF1 * member_size)
register('A6·SCu', 'T_spatial, structure-level global test', 'T_spatial (structures as replicates)', 'self-contained, structure level',
         'N', {p: np.array(pathways)[bh(v) <= ALPHA] for p, v in scu.items()},
         evidence=-np.log10(pd.Series(np.clip(scu['species'], 1e-300, None), index=pathways)),
         runtime=time.perf_counter() - t0, demo=True, returns=('yes/no', 'direction', 'location', 'shape', 'genes'))

### 3.6 · A8, A9 and A10: tradeSeq-style tests and the specimen-level gene F

- **A8 (association per group).** Within each group, a gene's position dependence (spline against
  specimen offsets) is tested and the pathways are called by the joint test. A pathway is
  "differential" when it is called in exactly one group: the difference-in-significance design.
- **A9 (start versus end).** \|Δ(end) − Δ(start)\| of the group-difference curve, with the joint
  test.
- **A10.** The specimen-level split-plot gene F, with notebook 37's matched test and joint test, and
  preranked GSEA.

In [ ]:


def matched_task(statistic, gene_names, sets, strata_):
    """Matched rank-AUC against 9,999 covariate-matched random sets (seed 12) for one statistic."""
    frame = matched_pathway_tests(pd.DataFrame({'statistic': statistic}, index=gene_names), sets, strata_,
                                  n_null=N_NULL, seed=SEED).set_index('pathway_id').reindex(list(sets))
    return {c: frame[c].to_numpy(float) for c in ('auc', 'null_auc_mean', 'null_auc_sd')}


def association_task(y, s, spec, knots_, gene_names, sets, strata_):
    """tradeSeq-style association in one group: spline against specimen offsets (partial F), then the matched test."""
    names = np.unique(spec)
    nuisance = np.column_stack([(spec == n).astype(float) - (spec == names[-1]).astype(float) for n in names[:-1]])
    designs = {'offset': np.column_stack([np.ones(len(s)), nuisance]),
               'spline': np.column_stack([make_gam_design(s, knots_), nuisance])}
    f_stat = nested_partial_f(y, designs, np.ones(len(s)), {'F': ('offset', 'spline')})['F']
    return {'F': f_stat, **matched_task(f_stat, gene_names, sets, strata_)}


def joint_from(result):
    frame = pd.DataFrame({'pathway_id': pathways, 'statistic': 'a', **{c: result[c] for c in ('auc', 'null_auc_mean', 'null_auc_sd')}})
    tests = joint_matched_test(frame.assign(effect=frame.auc - .5), VIF, keys=('statistic',)).set_index('pathway_id')
    tests['called'] = tests.effect.gt(0) & tests.q_joint.le(ALPHA)
    return tests


association_tasks = {}
for p in PARTS:
    in_pair = np.isin(specimen, partitions[p])   # group 1 = the partition's listed pair (species split: the mice)
    for g, rows in (('group1', in_pair), ('group0', ~in_pair)):
        association_tasks[f'{p}|{g}'] = (inputs['Y'][rows], position[rows], specimen[rows], knots, genes, gene_sets, strata)
association_runs = cached_tasks('association', association_task, association_tasks,
                                params={'n_null': N_NULL, 'seed': SEED, 'groups': {p: partitions[p] for p in PARTS}},
                                inputs={**input_key, 'sets': gene_sets, 'strata': strata})
association_calls, association_z = {}, {}
for p in PARTS:
    called = {g: set(joint_from(association_runs[f'{p}|{g}'][0]).pipe(lambda t: t.index[t.called])) for g in ('group1', 'group0')}
    association_calls[p] = called['group1'] ^ called['group0']
    if p == 'species':
        association_z = {g: joint_from(association_runs[f'species|{g}'][0]).z_joint.reindex(pathways) for g in ('group1', 'group0')}
sec_assoc = sum(sec for _, sec in association_runs.values())
register('A8·assoc', 'tradeSeq-style association per group, lists compared', 'per-group association F', 'joint test per group',
         'E', association_calls, evidence=(association_z['group1'] - association_z['group0']).abs(),
         direction=np.sign(association_z['group1'] - association_z['group0']), runtime=sec_assoc, returns=('yes/no', 'direction'))

curves = {p: newest(nb56 / 'stage_cache', 'curves_' + slug(p)) for p in PARTS}
np.testing.assert_allclose(curves['species']['T_spatial'], gene37, rtol=1e-6)   # Guard: notebook 56's Δ curves
sve_stat = {p: np.abs(curves[p]['delta'][:, -1] - curves[p]['delta'][:, 0]) for p in PARTS}
sve_runs = cached_tasks('start_vs_end', matched_task, {p: (sve_stat[p], genes, gene_sets, strata) for p in PARTS},
                        params={'n_null': N_NULL, 'seed': SEED}, inputs={'stat': [sve_stat[p] for p in PARTS], 'sets': gene_sets,
                                                                         'strata': strata})
sve = {p: joint_from(r) for p, (r, _) in sve_runs.items()}
sec_sve = sum(sec for _, sec in sve_runs.values())
register('A9·ends', 'tradeSeq-style start vs end, joint test', '|Δ(end) − Δ(start)|', 'joint test', 'E',
         {p: t.index[t.called] for p, t in sve.items()}, evidence=sve['species'].z_joint, runtime=sec_sve,
         returns=('yes/no', 'location'))

split_matched = {p: newest(nb37 / 'stage_cache', f'split_matched_{slug(p)}', kind='csv').set_index('pathway_id') for p in PARTS}
register('A10·MAUC', 'Split-plot gene F, matched rank-AUC', 'specimen-level split-plot F', 'matched rank-AUC', 'U',
         {p: t.index[t.effect.gt(0) & t.q_empirical.le(ALPHA)] for p, t in split_matched.items()},
         evidence=-np.log10(split_matched['species'].p_empirical), runtime=LOADED, returns=('yes/no',))
diag = pd.read_csv(nb37 / 'tables' / 'split_plot_diagnostics.csv', index_col=0)['0']
assert [len(CALLS['A10·MAUC'][p]) for p in PARTS] == [int(diag.iloc[0]), int(diag.iloc[1]), int(diag.iloc[2])]   # Guard
summary37 = pd.read_csv(nb37 / 'tables' / 'split_plot_summary.csv', index_col=0)['0']
assert [int(summary37.iloc[0]), int(summary37.iloc[1]), int(summary37.iloc[2])] == [0, 0, 0]
register('A10·JNT', 'Split-plot gene F, joint test (pseudobulk VIF)', 'specimen-level split-plot F', 'joint test', 'U',
         {p: [] for p in PARTS}, runtime=LOADED, returns=('yes/no',), note='notebook 37: 0, 0, 0 calls; VIF overcorrects')
calls, evid, sec = gsea_method('split_f', lambda p: split37[p]['F'])
register('A10·GSEA', 'Split-plot gene F, preranked GSEA', 'specimen-level split-plot F', 'preranked GSEA', 'N', calls,
         evidence=evid, runtime=sec, returns=('yes/no', 'genes'))

## 4 · Family C · score-first: one activity score per pathway and structure, then a positional test

**Idea.** Turn each pathway into one number per structure, an activity score, and ask whether its
curve along the PT differs in shape between species.

- **What it can answer.**
  - It shows each specimen's activity curve directly.
  - Its test is self-contained: "does this pathway's activity curve differ?" No other genes are
    involved.
- **What it cannot answer.**
  - Whether the pathway differs **more than other genes**. With a whole-tissue species difference,
    almost every pathway's activity differs, so the calls do not rank pathways.
  - Genes changing in opposite directions cancel in a score. That is why the program heatmaps of
    section 8 show genes, not scores.

Five scores, each tested two ways:

| Score | Definition |
|---|---|
| mean-z | the members' mean of species-balanced z-scored expression (notebooks 31, 56) |
| AUCell-like | the members' recovery of each structure's top 5% genes (notebook 56) |
| ssGSEA-like | the closed-form ssGSEA enrichment, Σ_S R^1.25 / Σ_S R^0.25 − Σ_rest R / (G − n), with R the within-structure expression rank |
| PLAGE | the first principal component of the members' z-scored expression (species-balanced weights), signed to agree with mean-z |
| singscore-like | the members' mean within-structure rank ÷ G, minus 0.5 |

The two tests are:
- **GAM.** The structure-level species × spline partial F with F(df) and BH. This is notebook 31's
  pseudoreplicated test (12,000 structures as replicates).
- **SP.** The specimen × 8-bin split plot: cubic species × position judged against specimen ×
  position, with a moderated F and BH (notebook 37's error term).

Mean-z by GAM is notebook 31's strategy 3b (U). Mean-z and AUCell-like by SP are the earlier
notebook-61 rows (E). The other seven combinations are new.

In [ ]:
scores = newest(nb56 / 'stage_cache', 'structure_scores')
taxonomy56 = pd.read_csv(nb56 / 'tables' / 'f_taxonomy.csv').set_index('approach')
weights = np.zeros(len(position))
for sp_, names_ in SPECIMENS.items():
    for n in names_:
        rows = specimen == n
        weights[rows] = len(position) / (2 * len(names_) * rows.sum())


ROW_CHUNKS, SET_CHUNKS = 24, 32   # fixed task layout (independent of the number of workers)


def rank_score_task(block, mt, n):
    """ssGSEA-like and singscore-like scores for a block of structures (average ranks within each structure)."""
    ranks = rankdata(block, axis=1).astype(np.float32)          # top-expressed gene = G
    G = block.shape[1]
    sum_r = ranks @ mt
    ssgsea = (ranks ** (1 + SSGSEA_ALPHA)) @ mt / ((ranks ** SSGSEA_ALPHA) @ mt) - (G * (G + 1) / 2 - sum_r) / (G - n)
    return {'ssgsea': ssgsea.astype(np.float32), 'singscore': (sum_r / n / G - .5).astype(np.float32)}


def plage_task(z, root, columns):
    """PLAGE: first principal component of the members' z-scores (species-balanced weights), signed to agree with mean-z."""
    out = np.empty((len(z), len(columns)), np.float32)
    for j, idx in enumerate(columns):
        zs = z[:, idx]
        _, vectors = np.linalg.eigh((root * zs).T @ (root * zs))
        score = zs @ vectors[:, -1]
        out[:, j] = score if np.corrcoef(score, zs.mean(axis=1))[0, 1] >= 0 else -score
    return {'plage': out}


member_index = {p: pd.Index(genes).get_indexer(v) for p, v in gene_sets.items()}
dense = inputs['Y'].toarray()
mt = member_matrix.T.toarray().astype(np.float32)
row_edges = np.linspace(0, len(dense), ROW_CHUNKS + 1).astype(int)
rank_runs = cached_tasks('scores_rank', rank_score_task,
                         {str(i): (dense[a:b], mt, member_size.astype(np.float32)) for i, (a, b) in enumerate(zip(row_edges[:-1], row_edges[1:]))},
                         params={'alpha': SSGSEA_ALPHA, 'chunks': ROW_CHUNKS}, inputs={'Y': input_key['Y'], 'sets': gene_sets})
w_norm = weights / weights.sum()
mean_w = w_norm @ dense
z_dense = (dense - mean_w) / np.sqrt(np.maximum(w_norm @ (dense - mean_w) ** 2, 1e-12))
del dense
set_edges = np.linspace(0, len(pathways), SET_CHUNKS + 1).astype(int)
plage_runs = cached_tasks('scores_plage', plage_task,
                          {str(i): (z_dense, np.sqrt(w_norm)[:, None], [member_index[p] for p in pathways[a:b]])
                           for i, (a, b) in enumerate(zip(set_edges[:-1], set_edges[1:]))},
                          params={'chunks': SET_CHUNKS}, inputs={'Y': input_key['Y'], 'sets': gene_sets, 'weights': digest(weights)})
del z_dense
new_scores = {'ssgsea': np.vstack([rank_runs[str(i)][0]['ssgsea'] for i in range(ROW_CHUNKS)]),
              'singscore': np.vstack([rank_runs[str(i)][0]['singscore'] for i in range(ROW_CHUNKS)]),
              'plage': np.hstack([plage_runs[str(i)][0]['plage'] for i in range(SET_CHUNKS)])}
sec_rank = sum(sec for _, sec in rank_runs.values())
sec_plage = sum(sec for _, sec in plage_runs.values())
SCORES = {'meanz': scores['mean_z'], 'aucell': scores['aucell'], **new_scores}
SCORE_LABELS = {'meanz': 'mean-z', 'aucell': 'AUCell-like', 'ssgsea': 'ssGSEA-like', 'plage': 'PLAGE',
                'singscore': 'singscore-like'}
SCORE_SECONDS = {'meanz': 0., 'aucell': 0., 'ssgsea': sec_rank / 2, 'singscore': sec_rank / 2, 'plage': sec_plage}
sample_rows, sample_sets = np.arange(0, len(position), 7), np.arange(0, len(pathways), 15)
print('Median Spearman correlation with mean-z (every 7th structure, every 15th pathway):',
      {SCORE_LABELS[k]: round(float(np.median([spearmanr(SCORES['meanz'][sample_rows, j], v[sample_rows, j])[0] for j in sample_sets])), 2)
       for k, v in SCORES.items() if k != 'meanz'})


def score_gam_task(matrix, p, group, nuisance):
    """Notebook 31's structure-level shape test of a score: species x spline beyond an offset, F(df)."""
    designs, w = contrast_designs(position, group, specimen, knots, nuisance_shape=nuisance)
    result = nested_partial_f(np.asarray(matrix, float), designs, w, {'T_spatial': ('level', 'full')})
    return {'p': f_dist.sf(result['T_spatial'], *result['df']['T_spatial'])}


lo, hi = grid[0], grid[-1]
bins = np.clip(((position - lo) / (hi - lo) * N_BINS).astype(int), 0, N_BINS - 1)
cell_sizes = pd.Series(1, index=pd.MultiIndex.from_arrays([specimen, bins])).groupby(level=[0, 1]).size()
cell_keys = cell_sizes[cell_sizes >= MIN_CELL].index


def split_plot(matrix):
    """Specimen x 8-bin means, cubic species x position against specimen x position (moderated F)."""
    means = pd.DataFrame(np.asarray(matrix, float)).groupby([specimen, bins]).mean().loc[cell_keys]
    cell_spec = means.index.get_level_values(0).to_numpy()
    basis = make_gam_design((means.index.get_level_values(1).to_numpy() + .5) / N_BINS, np.array([]))[:, 1:]
    spec_cols = (cell_spec[:, None] == np.array(ALL_SPECIMENS)[None, :]).astype(float)
    cell_human = np.isin(cell_spec, SPECIMENS['human']).astype(float)
    out = {}
    for p in PARTS:
        if p == 'species':
            design = np.column_stack([spec_cols, basis, cell_human[:, None] * basis])
        else:
            group = np.isin(cell_spec, partitions[p]).astype(float)
            design = np.column_stack([spec_cols, basis, cell_human[:, None] * basis, group[:, None] * basis])
        test = list(range(design.shape[1] - basis.shape[1], design.shape[1]))
        out[p] = moderated_f(means.to_numpy(float), np.ones(means.shape), design, test)['p']
    return out


def group_of(p):
    return (human.astype(float), None) if p == 'species' else (np.isin(specimen, partitions[p]).astype(float), human.astype(float))


score_tasks = {f'{name}|GAM|{p}': (score_gam_task, (matrix, p, *group_of(p)))
               for name, matrix in SCORES.items() for p in PARTS if name != 'meanz' or p == 'species'}
score_tasks.update({f'{name}|SP|all': (lambda m: {f'p{i}': v for i, v in enumerate(split_plot(m).values())}, (matrix,))
                    for name, matrix in SCORES.items()})
score_runs = cached_tasks('score_tests', lambda function, args: function(*args), {k: v for k, v in score_tasks.items()},
                          params={'parts': PARTS, 'bins': N_BINS, 'min_cell': MIN_CELL},
                          inputs={'scores': {k: digest(v) for k, v in SCORES.items()}, **input_key})
h = hits31[hits31.strategy.eq('3b · Pathway score GAM, shape')]
calls_meanz_gam = {p: h.pathway_id[h.partition.eq(p) & h.hit] for p in PARTS}
assert set(np.array(pathways)[bh(score_runs['meanz|GAM|species'][0]['p']) <= ALPHA]) == set(calls_meanz_gam['species'])   # Guard: notebook 31
status_c = {('meanz', 'GAM'): 'U', ('meanz', 'SP'): 'E', ('aucell', 'SP'): 'E'}
for name in SCORES:
    for test in ('GAM', 'SP'):
        key = f'C·{name}·{test}'
        if test == 'GAM':
            parts = ['species'] if name == 'meanz' else PARTS
            pvalues = {p: score_runs[f'{name}|GAM|{p}'][0]['p'] for p in parts}
            sec = sum(score_runs[f'{name}|GAM|{p}'][1] for p in parts)
        else:
            run, sec = score_runs[f'{name}|SP|all']
            pvalues = {p: run[f'p{i}'] for i, p in enumerate(PARTS)}
        calls = calls_meanz_gam if key == 'C·meanz·GAM' else {p: np.array(pathways)[bh(v) <= ALPHA] for p, v in pvalues.items()}
        runtime = LOADED if key == 'C·meanz·GAM' else sec + SCORE_SECONDS[name] / 2   # score computation shared by both tests
        register(key, f'{SCORE_LABELS[name]} score, {"structure-level GAM" if test == "GAM" else "specimen × bin split plot"}',
                 f'{SCORE_LABELS[name]} score per structure', 'GAM shape F (structures)' if test == 'GAM' else 'split-plot moderated F',
                 status_c.get((name, test), 'N'), calls,
                 evidence=pd.Series(-np.log10(np.clip(pvalues['species'], 1e-300, None)), index=pathways),
                 runtime=runtime, returns=('yes/no', 'shape'))
check_counts('C·meanz·GAM', '3b · Pathway-score GAM, shape')
for key, label in (('C·meanz·SP', 'Pathway activity (mean-z module score), specimen × bin split plot'),
                   ('C·aucell·SP', 'Pathway activity (AUCell-like score), specimen × bin split plot')):
    assert len(CALLS[key]['species']) == int(taxonomy56.loc[label, 'species calls'])   # Guard: notebook 56

## 5 · Family B · gene-first: cluster genes into modules, then annotate the modules with pathways

**Idea.** Let the data group the genes first, then ask which pathways fill each group. A module is a
set of genes with a shared pattern. Pathways enter only afterwards, by over-representation (ORA) in
each module.

- **What it can answer.**
  - Which **shapes** of species difference recur, and which genes share them, whatever library they
    come from.
  - It can find a coherent gene group that no library pathway describes.
- **What it cannot answer.**
  - Whether a pathway as a whole differs. A pathway is "called" only when enough of its members fall
    in one module, so a pathway whose members change in two different ways is split.
  - The module ORA is competitive against the selected genes, not against matched genes. Expression,
    detection and coverage are not matched.
- **Gene-first against pathway-first.**
  - Pathway-first tests each library set and then looks at its genes.
  - Gene-first finds the gene groups and then looks up the library.
  - When both work they should point at the same genes. The reverse question in section 6 checks
    this against the primary method's programs.

Five constructions:

| Code | Genes clustered | Pattern clustered | How a species call arises |
|---|---|---|---|
| B1 (E) | top 10% by T_spatial | standardised Δ(s), k-means, k = 8 (notebook 56) | pathways enriched in any module |
| B2 (N) | split-plot significant genes (BH ≤ 0.05), a calibrated selection | standardised Δ(s), k-means | as B1; a partition with < 10·k such genes has no modules |
| B3 (N) | all 11,071 genes | standardised Δ(s), k-means | as B1 |
| B4 (N) | top 10% by T_total (offset or shape) | the two groups' curves, each z-scored, concatenated | as B1 |
| B5 (N) | the 3,000 most variable genes (within specimen), species-blind | WGCNA-like co-expression across structures | modules whose eigengene curve differs in the split plot, then their enriched pathways |

**Choosing k and treating the relabelings.**
- For B2–B4, k is chosen among 3–8 by mean resampling stability: the ARI over 20 resamples of 80%,
  on the species run. It is then frozen for the relabelings.
- B1–B4 define modules **from the species difference**, so a module as a unit is circular. Their
  calls are the annotated pathways (as in notebook 56). The relabelings recluster their own
  differences.
- B5 is species-blind. Its modules are fixed once, and only the eigengene test changes with the
  partition.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score


def standardize(d):
    return (d - d.mean(axis=1, keepdims=True)) / np.maximum(d.std(axis=1, keepdims=True), 1e-12)


def kmeans_task(data, k, seed, pick):
    """One k-means fit (20 starts) on all rows (``pick`` empty) or on the picked rows."""
    rows = data if not len(pick) else data[pick]
    return {'labels': KMeans(n_clusters=k, n_init=KM_STARTS, random_state=seed).fit(rows).labels_}


def choose_k(stage, data):
    """Notebook 56's rule: mean ARI of 20 resamples of 80% per k; the most stable k, ties to the smaller.

    Resamples are drawn in the parent from one fixed stream (seed 56, as notebook 56) and each fit has
    its own fixed seed, so the result does not depend on the number of workers.
    """
    rng = np.random.default_rng(KM_SEED)
    tasks = {}
    for k in K_RANGE:
        tasks[f'{k}|full'] = (data, k, KM_SEED, np.array([], int))
        for r in range(N_RESAMPLE):
            pick = np.sort(rng.choice(len(data), int(.8 * len(data)), replace=False))
            tasks[f'{k}|{r}'] = (data, k, KM_SEED + r + 1, pick)
    runs = cached_tasks(stage, kmeans_task, tasks, params={'k': list(K_RANGE), 'resamples': N_RESAMPLE, 'seed': KM_SEED,
                                                           'starts': KM_STARTS}, inputs={'data': digest(data)})
    stability = {}
    for k in K_RANGE:
        full = runs[f'{k}|full'][0]['labels']
        stability[k] = float(np.mean([adjusted_rand_score(full[tasks[f'{k}|{r}'][3]], runs[f'{k}|{r}'][0]['labels'])
                                      for r in range(N_RESAMPLE)]))
    best = max(stability, key=lambda k: (round(stability[k], 6), -k))
    return best, stability, sum(sec for _, sec in runs.values())


def module_ora(selected_rows, labels, k, matrix):
    """ORA p per pathway and module (pathways x modules) against ``matrix``'s universe."""
    cols = []
    for m in range(k):
        chosen = np.zeros(matrix.shape[1], bool)
        chosen[selected_rows[labels == m]] = True
        cols.append(ora_pvalues(chosen, matrix))
    return np.column_stack(cols)


MODULE_GENES, MODULE_TABLES = {}, {}


def curve_modules(key, label, status, data_of, k_fixed=None, min_per_module=0, returns=('yes/no', 'shape', 'genes')):
    """B1-B4: modules of each partition's selected genes, annotated by ORA; calls = pathways enriched in any module."""
    rows_species, data_species = data_of('species')
    if k_fixed is None:
        k, stability, sec_k = choose_k(f'kselect_{key[:2]}', data_species)
        print(f'{key}: mean resample ARI by k', {kk: round(v, 3) for kk, v in stability.items()}, '→ k =', k)
    else:
        k, sec_k = k_fixed, 0.
    prepared = {p: data_of(p) for p in PARTS}
    fit = {p: v for p, v in prepared.items() if len(v[0]) >= max(min_per_module * k, k + 1)}
    runs = cached_tasks(f'modules_{key[:2]}', kmeans_task, {p: (fit[p][1], k, KM_SEED, np.array([], int)) for p in fit},
                        params={'k': k, 'seed': KM_SEED, 'starts': KM_STARTS}, inputs={p: digest(fit[p][1]) for p in fit})
    calls, evidence = {}, None
    t0 = time.perf_counter()
    for p in PARTS:
        if p not in fit:
            calls[p] = []
            print(f'{key} {p}: {len(prepared[p][0])} genes selected, fewer than {min_per_module} × k, so no modules')
            continue
        labels = runs[p][0]['labels']
        pv = module_ora(fit[p][0], labels, k, member_matrix)
        q = bh(pv.ravel()).reshape(pv.shape)
        calls[p] = np.array(pathways)[(q <= ALPHA).any(axis=1)]
        if p == 'species':
            evidence = pd.Series(-np.log10(pv.min(axis=1).clip(1e-300)), index=pathways)
            MODULE_GENES[key] = [list(np.array(genes)[fit[p][0][labels == m]]) for m in range(k)
                                 if (q[:, m] <= ALPHA).any()]
            MODULE_TABLES[key] = pd.DataFrame({'module': range(1, k + 1), 'genes': np.bincount(labels, minlength=k),
                                               'enriched pathways': (q <= ALPHA).sum(axis=0)})
    seconds = sec_k + sum(sec for _, sec in runs.values()) + time.perf_counter() - t0
    register(key, label, 'Δ(s) curves' if key[:2] != 'B4' else 'group curves', 'modules + ORA', status, calls,
             evidence=evidence, runtime=seconds, returns=returns, note=f'k = {k}')
    return None


def top_rows(values, share=TOP_SHARE):
    return np.sort(np.argsort(-values)[:int(round(share * len(values)))])


# B1: notebook 56's construction, k = 8 (guard: notebook 56's calls).
k_b1, _, _ = choose_k('kselect_B1', standardize(curves['species']['delta'][top_rows(curves['species']['T_spatial'])]))
assert k_b1 == K_MODULES, 'notebook 56 chose k = 8'
curve_modules('B1·T10', f'Curve modules, top 10% T_spatial (k = {K_MODULES}) + ORA', 'E',
              lambda p: (top_rows(curves[p]['T_spatial']), standardize(curves[p]['delta'][top_rows(curves[p]['T_spatial'])])),
              k_fixed=K_MODULES)
row56 = taxonomy56.loc[f'Curve modules (k = {K_MODULES}) + ORA']
assert len(CALLS['B1·T10']['species']) == int(row56['species calls'])   # Guard: notebook 56
assert [len(CALLS['B1·T10'][s]) for s in SWAPS] == [int(x) for x in row56['relabeled calls'].split(', ')]


def split_selected(p):
    rows = np.flatnonzero(bh(split37[p]['p']) <= ALPHA)
    return rows, standardize(curves[p]['delta'][rows])


curve_modules('B2·split', 'Curve modules, split-plot significant genes + ORA', 'N', split_selected, min_per_module=10)
curve_modules('B3·all', 'Curve modules, all genes + ORA', 'N',
              lambda p: (np.arange(len(genes)), standardize(curves[p]['delta'])))

descriptors58 = newest(nb58_cache, 'descriptors')
assert np.allclose(descriptors58['grid'], grid), 'notebook 58 curves are on another grid'
specimen_curves58 = {n: descriptors58[f'{n}|curves'] for n in ALL_SPECIMENS}


def concatenated(p):
    rows = top_rows(gene_scores37[p]['T_total'])
    first = np.mean([specimen_curves58[n][rows] for n in partitions[p]], axis=0)
    second = np.mean([specimen_curves58[n][rows] for n in ALL_SPECIMENS if n not in partitions[p]], axis=0)
    return rows, np.hstack([standardize(first), standardize(second)])


curve_modules('B4·concat', 'Concatenated group curves, top 10% T_total + ORA', 'N', concatenated)

### B5 · WGCNA-like species-blind modules, tested as units

The construction:
- Expression is centred within each specimen, so the modules capture co-variation along and across
  structures, not specimen offsets.
- Signed adjacency is ((1 + r)/2)⁶ and distance is 1 − adjacency. Average linkage is cut at height
  0.9, and modules need ≥ 30 genes.
- There is no topological overlap, which simplifies WGCNA.
- Each module's eigengene (its first principal component, signed to agree with the module mean) is
  averaged in specimen × 8-bin cells and tested in the split plot under each partition.

Called modules contribute their enriched pathways (ORA within the 3,000 genes).

In [ ]:


def wgcna_modules():
    y = inputs['Y']
    within = np.zeros(len(genes))
    for n in ALL_SPECIMENS:
        block = y[specimen == n]
        mean = np.asarray(block.mean(axis=0)).ravel()
        within += (np.asarray(block.multiply(block).mean(axis=0)).ravel() - mean ** 2) / len(ALL_SPECIMENS)
    chosen = np.sort(np.argsort(-within)[:WGCNA_GENES])
    x = y[:, chosen].toarray()
    for n in ALL_SPECIMENS:
        rows = specimen == n
        x[rows] -= x[rows].mean(axis=0)
    x /= np.maximum(x.std(axis=0), 1e-12)
    r = np.corrcoef(x.T)
    distance = 1 - ((1 + r) / 2) ** WGCNA_POWER
    np.fill_diagonal(distance, 0)
    raw = fcluster(linkage(squareform(np.clip(distance, 0, None), checks=False), 'average'), WGCNA_HEIGHT, criterion='distance')
    sizes = pd.Series(raw).value_counts()
    kept = [c for c in sizes.index if sizes[c] >= WGCNA_MIN]           # by size, ties by first appearance
    labels = np.full(len(chosen), -1)
    eigengenes = np.empty((len(x), len(kept)))
    for m, c in enumerate(kept):
        members = raw == c
        labels[members] = m
        u, sv, _ = np.linalg.svd(x[:, members], full_matrices=False)
        e = u[:, 0] * sv[0]
        eigengenes[:, m] = e if np.corrcoef(e, x[:, members].mean(axis=1))[0, 1] >= 0 else -e
    return {'genes': chosen, 'labels': labels, 'eigengenes': eigengenes}


b5, sec_b5 = timed('wgcna', wgcna_modules, kind='payload',
                   params={'genes': WGCNA_GENES, 'power': WGCNA_POWER, 'height': WGCNA_HEIGHT, 'min': WGCNA_MIN}, inputs=input_key)
n_modules = b5['eigengenes'].shape[1]
t0 = time.perf_counter()
eigengene_p = split_plot(b5['eigengenes'])
module_called = {p: bh(v) <= ALPHA for p, v in eigengene_p.items()}
b5_universe = list(np.array(genes)[b5['genes']])
b5_matrix = membership(gene_sets, b5_universe)
b5_ora = module_ora(np.arange(len(b5_universe)), b5['labels'], n_modules, b5_matrix)
b5_q = bh(b5_ora.ravel()).reshape(b5_ora.shape)
b5_calls = {p: np.array(pathways)[(b5_q[:, module_called[p]] <= ALPHA).any(axis=1)] if module_called[p].any() else []
            for p in PARTS}
MODULE_GENES['B5·wgcna'] = [list(np.array(b5_universe)[b5['labels'] == m]) for m in range(n_modules) if module_called['species'][m]]
B5_TABLE = pd.DataFrame({'module': range(1, n_modules + 1), 'genes': np.bincount(b5['labels'][b5['labels'] >= 0], minlength=n_modules),
                         'q species': bh(eigengene_p['species']), **{f'q {s}': bh(eigengene_p[s]) for s in SWAPS},
                         'enriched pathways': (b5_q <= ALPHA).sum(axis=0),
                         'A6·JNT calls among them': [len(set(np.array(pathways)[b5_q[:, m] <= ALPHA]) & CALLS['A6·JNT']['species'])
                                                    for m in range(n_modules)],
                         'top pathways': ['; '.join(NAMES[np.array(pathways)[np.argsort(b5_ora[:, m])[:3]]]) for m in range(n_modules)]})
register('B5·wgcna', 'WGCNA-like species-blind modules, eigengene split plot + ORA', 'co-expression modules (species-blind)',
         'eigengene split plot, then ORA', 'N', b5_calls,
         evidence=pd.Series(-np.log10(b5_ora.min(axis=1).clip(1e-300)), index=pathways),
         runtime=sec_b5 + time.perf_counter() - t0, returns=('yes/no', 'genes'),
         note=f'{n_modules} modules; called for species {int(module_called["species"].sum())}, '
              f'relabelings {", ".join(str(int(module_called[s].sum())) for s in SWAPS)}')
B5_TABLE.to_csv(tables / 'b5_modules.csv', index=False)
display(B5_TABLE.round(4))
pd.concat(MODULE_TABLES, names=['method']).to_csv(tables / 'curve_modules.csv')

## 6 · The common comparison

### 6.0 · Collapsing redundant pathways into programs

Reactome, KEGG and Hallmark contain many near-duplicate pathways, so a call count partly counts
library redundancy.

**Primary rule (notebook 37, SM17).** Pathways are grouped by the genes that carry their signal:
- those genes are the members in the top 10% of genes by T_spatial, or all members when fewer than
  three qualify;
- grouping is average linkage on 1 − overlap coefficient, cut at 0.5;
- it uses notebook 37's function;
- **order-invariant** (plan addendum 3): average linkage with tied distances depends on input
  order, so pathways are sorted by `pathway_id` before clustering.

The 36 robust pathways are regrouped and compared with notebook 37's 17 programs, which were made in
library order and are superseded by this rule.

**Membership check.** The same linkage and cut on 1 − Jaccard of full membership. The rules were fixed
in `plan.md` before this ran.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import adjusted_rand_score
from pseudospace.pathway_decoys import overlap_programs

PROGRAM_CUT = .5
contributing = set(gene37.index[gene37.ge(gene37.quantile(1 - TOP_SHARE))])


def driver_programs(ids):
    """Order-invariant programs: notebook 37's rule with pathways sorted by pathway_id before clustering.

    Average linkage with tied distances depends on input order; a fixed (sorted) order makes the result
    canonical. Programs are numbered by size, ties by the smallest member id.
    """
    ids = sorted(set(ids))
    if not ids:
        return pd.Series(dtype=int)
    return overlap_programs(gene_sets, ids, contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program


def membership_programs(ids):
    """The same linkage and cut on 1 - Jaccard of full membership (a check), sorted by pathway_id."""
    ids = sorted(set(ids))
    if len(ids) < 2:
        return pd.Series(1, index=ids, dtype=int)
    members = [set(gene_sets[p]) for p in ids]
    d = np.zeros((len(ids), len(ids)))
    for i in range(len(ids)):
        for j in range(i):
            d[i, j] = d[j, i] = 1 - len(members[i] & members[j]) / len(members[i] | members[j])
    raw = fcluster(linkage(squareform(d), 'average'), PROGRAM_CUT, criterion='distance')
    relabel = {old: new for new, old in enumerate(pd.Series(raw).value_counts().index, start=1)}
    return pd.Series([relabel[r] for r in raw], index=ids)


def partition(labels):
    return {frozenset(block.index) for _, block in labels.groupby(labels)}


robust36 = table37.index[table37.robust.astype(bool)]
library_order = overlap_programs(gene_sets, [p for p in pathways if p in set(robust36)], contributing=contributing,
                                 cut=PROGRAM_CUT).set_index('pathway_id').program
assert partition(library_order) == partition(table37.loc[robust36, 'program'].astype(int))   # Guard: notebook 37's rule
invariant36 = driver_programs(robust36)
old37 = table37.loc[robust36, 'program'].astype(int)


def comembers(labels, p):
    return frozenset(labels.index[labels.eq(labels[p])]) - {p}


moved = [p for p in sorted(robust36) if comembers(invariant36, p) != comembers(old37, p)]
MOVES = pd.DataFrame({'pathway': NAMES[moved], 'notebook 37 program': old37[moved],
                      'notebook 37 co-members': [', '.join(sorted(NAMES[list(comembers(old37, p))])) for p in moved],
                      'order-invariant program': invariant36[moved],
                      'order-invariant co-members': [', '.join(sorted(NAMES[list(comembers(invariant36, p))])) for p in moved]})
MOVES.to_csv(tables / 'robust36_program_moves.csv')
print(f'Robust 36: notebook 37 (library order) {old37.nunique()} programs; order-invariant {invariant36.nunique()} programs; '
      f'{len(moved)} pathways change co-members.')
display(MOVES)


def program_names(labels, score):
    """Name each program after its member with the highest score (ties by pathway id)."""
    names = {}
    for program, block in labels.groupby(labels):
        best = score.reindex(block.index).fillna(-np.inf).sort_index().sort_values(ascending=False, kind='stable').index[0]
        names[program] = NAMES[best]
    return labels.map(names)


def disagreements(ids):
    a, b = driver_programs(ids), membership_programs(ids)
    rows = []
    for i, p in enumerate(sorted(ids)):
        for q in sorted(ids)[i + 1:]:
            same_a, same_b = a[p] == a[q], b[p] == b[q]
            if same_a != same_b:
                rows.append({'pathway 1': NAMES[p], 'pathway 2': NAMES[q],
                             'grouped by drivers': same_a, 'grouped by membership': same_b,
                             'contributing overlap coefficient': len(set(gene_sets[p]) & set(gene_sets[q]) & contributing)
                             / max(min(len(set(gene_sets[p]) & contributing), len(set(gene_sets[q]) & contributing)), 1),
                             'membership Jaccard': len(set(gene_sets[p]) & set(gene_sets[q]))
                             / len(set(gene_sets[p]) | set(gene_sets[q]))})
    return pd.DataFrame(rows), adjusted_rand_score(a.sort_index(), b.sort_index())


primary_list = pd.read_csv(nb50 / 'primary_pathway_list.csv')
primary_ids = primary_list.pathway_id.tolist()
PROGRAM_SETS = {'primary 22': primary_ids, 'A6·JNT calls (60)': sorted(CALLS['A6·JNT']['species'])}
program_summary = []
for label, ids in PROGRAM_SETS.items():
    drivers, members = driver_programs(ids), membership_programs(ids)
    table_dis, ari = disagreements(ids)
    table_dis.to_csv(tables / f"program_rule_disagreements_{label.split(' ')[0].lower()}.csv", index=False)
    program_summary.append({'set': label, 'pathways': len(ids), 'programs (driver rule)': drivers.nunique(),
                            'programs (membership check)': members.nunique(),
                            'pathway pairs grouped differently': len(table_dis), 'adjusted Rand index': ari})
    print(f'{label}: pairs grouped differently by the two rules')
    display(table_dis.round(2))
display(pd.DataFrame(program_summary).round(2))

primary_drivers = driver_programs(primary_ids)
primary_table = primary_list.copy()
primary_table['program'] = primary_table.pathway_id.map(primary_drivers)
primary_table['program name'] = primary_table.pathway_id.map(program_names(primary_drivers, EVIDENCE['A6·JNT']))
primary_table['membership-check program'] = primary_table.pathway_id.map(membership_programs(primary_ids))
primary_table.to_csv(tables / 'primary22_programs.csv', index=False)
c1_drivers = driver_programs(CALLS['A6·JNT']['species'])
c1_programs = pd.DataFrame({'program': c1_drivers, 'program name': program_names(c1_drivers, EVIDENCE['A6·JNT']),
                            'pathway': NAMES[c1_drivers.index]})
c1_programs.to_csv(tables / 'c1_calls_programs.csv')
display(primary_table[['name', 'program', 'program name', 'membership-check program']].sort_values('program'))

### 6.1 · External replication of each specific method's calls

Notebook 50's construction is reused:
- **References.** Census and Lake/KPMP human cortex donors, each against Census male mice.
- **Scorer.** The joint replication z of each pathway: the matched rank-AUC z of the external
  statistic, divided by √VIF from the donor-level residual correlations.
- **Null.** 999 within-strata gene relabelings applied to every set at once, so overlaps between
  called pathways are kept.
- **Statistics.**
  - Positional families use the signed species × (S3 − early) statistic.
  - Average-level rows (A1–A3) use the species main effect over the whole PT, signed by our whole-PT
    log2 fold change.

A method's result is the mean replication z of its species calls and the share of relabelings
reaching it (p). It is run for every specific method with at least five calls.

The permutations run in 27 chunks of 37, and each chunk is seeded by its own id. The p-values
therefore do not depend on the number of workers, though they differ from a single-stream run.

In [ ]:
from pseudospace.pathway_decoys import welch_t
from pseudospace.pathway_remodeling import matching_strata
from pseudospace.set_collection_null import JointReplicationZ, stratified_permutation, upper_p

covariates = inputs['covariates']


def census_donors(label, segment_map, early_parts):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_CELLS_EXTERNAL)]
    complete = meta.groupby('donor_id').segment.apply(lambda s: set(segment_map.values()).issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    late, early, whole, info = {}, {}, {}, []

    def log_cpm_of(index):
        return np.log2(table.loc[index].sum() / meta.loc[index, 'total_counts_all_genes'].sum() * 1e6 + 1)

    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        late_index, early_index = block.index[block.segment.eq('S3')], block.index[block.segment.isin(early_parts)]
        late[donor], early[donor] = log_cpm_of(late_index), log_cpm_of(early_index)
        whole[donor] = log_cpm_of(late_index.union(early_index))
        info.append({'donor': donor, 'sex': sex})
    frame = lambda values: pd.DataFrame(values).T.reindex(columns=genes)
    return frame(late), frame(early), frame(whole), pd.DataFrame(info).set_index('donor')


h_late, h_early, h_whole, _ = census_donors('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                                                     'epithelial cell of proximal tubule segment 3': 'S3'}, ['early'])
m_late, m_early, m_whole, m_info = census_donors('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                          'epithelial cell of proximal tubule segment 2': 'S2',
                                                          'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
males = m_info.index[m_info.sex.eq('male')]
lake = newest(nb50_cache, 'lake_pseudobulk')          # notebook 50's Lake/KPMP cortex pseudobulk, in our gene order
assert lake['sums'].shape[1] == len(genes)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups'].astype(str)], columns=['donor', 'sex', 'segment'])
lake_meta['nuclei'] = lake['nuclei']
nuclei = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei.index[(nuclei.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_CELLS_EXTERNAL).all(axis=1)]
lake_sums = pd.DataFrame(lake['sums'], columns=genes)


def lake_log_cpm(donor, parts):
    rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.isin(parts)]
    return np.log2(lake_sums.loc[rows].sum() / lake['totals'][rows].sum() * 1e6 + 1)


lake_late = pd.DataFrame({d: lake_log_cpm(d, ['S3']) for d in lake_donors}).T
lake_early = pd.DataFrame({d: lake_log_cpm(d, ['S1', 'S2']) for d in lake_donors}).T
lake_whole = pd.DataFrame({d: lake_log_cpm(d, ['S1', 'S2', 'S3']) for d in lake_donors}).T

short = pd.Series(segment).str.replace('PT-', '', regex=False).to_numpy()
ours_contrast = pd.DataFrame({n: np.asarray(inputs['Y'][(specimen == n) & (short == 'S3')].mean(axis=0)).ravel()
                              - np.asarray(inputs['Y'][(specimen == n) & np.isin(short, ['S1', 'S2'])].mean(axis=0)).ravel()
                              for n in ALL_SPECIMENS}, index=genes)
ours_raw = ours_contrast[SPECIMENS['human']].mean(axis=1) - ours_contrast[SPECIMENS['mouse']].mean(axis=1)


def positional_reference(human_late, human_early, mouse_late, mouse_early, mouse_donors):
    """Notebook 50's signed species x (S3 - early) reference (welch t across donors, signed by our cohort)."""
    hd = human_late - human_early
    md = (mouse_late - mouse_early).loc[mouse_donors]
    level_h = pd.concat([human_late.mean(), human_early.mean()], axis=1).max(axis=1)
    level_m = pd.concat([mouse_late.loc[mouse_donors].mean(), mouse_early.loc[mouse_donors].mean()], axis=1).max(axis=1)
    keep = (covariates[['detection_mouse', 'detection_human']].min(axis=1).ge(.10) & level_h.ge(1) & level_m.ge(1)
            & hd.notna().all() & md.notna().all())
    g = keep.index[keep.to_numpy()]
    centred = ours_contrast - ours_contrast.loc[g].median()
    ours_c = centred[SPECIMENS['human']].mean(axis=1) - centred[SPECIMENS['mouse']].mean(axis=1)
    hc, mc = hd.sub(hd[g].median(axis=1), axis=0), md.sub(md[g].median(axis=1), axis=0)
    t = pd.Series(welch_t(hc[g].to_numpy(), mc[g].to_numpy())[0], index=g)
    strata_g = matching_strata(pd.DataFrame({'mean_expression': ((level_h + level_m) / 2).loc[g].to_numpy(),
                                             'detection': covariates.loc[g, ['detection_mouse', 'detection_human']].min(axis=1).to_numpy(),
                                             'coverage': covariates.loc[g, 'coverage'].to_numpy()}), bins=3)
    sets_g = {p: [x for x in v if x in set(g)] for p, v in gene_sets.items()}
    sets_g = {p: v for p, v in sets_g.items() if 5 <= len(v) < len(g)}
    residuals = pd.concat([hc[g] - hc[g].mean(), mc[g] - mc[g].mean()])
    return {'genes': g, 'statistic': np.sign(ours_c.loc[g]) * t, 'strata': strata_g, 'sets': sets_g,
            'residuals': residuals.to_numpy()}


whole_lfc50 = newest(nb50_cache, 'deseq2_species', kind='csv').query("scope == 'whole_PT'").set_index('gene').log2FoldChange


def average_reference(info, human_whole, mouse_whole, mouse_donors):
    """Notebook 50's average-level reference: species main effect over the whole PT, signed by our whole-PT log2FC."""
    hw = human_whole.sub(human_whole[info].median(axis=1), axis=0)
    mw = mouse_whole.loc[mouse_donors].sub(mouse_whole.loc[mouse_donors, info].median(axis=1), axis=0)
    t_main = pd.Series(welch_t(hw[info].to_numpy(), mw[info].to_numpy())[0], index=info)
    sets_ = {p: [x for x in v if x in set(info)] for p, v in gene_sets.items()}
    sets_ = {p: v for p, v in sets_.items() if 5 <= len(v) < len(info)}
    residuals = pd.concat([hw[info] - hw[info].mean(), mw[info] - mw[info].mean()])
    strata_ = matching_strata(pd.DataFrame({'mean_expression': ((human_whole[info].mean() + mouse_whole.loc[mouse_donors, info].mean()) / 2).to_numpy(),
                                            'detection': covariates.loc[info, ['detection_mouse', 'detection_human']].min(axis=1).to_numpy(),
                                            'coverage': covariates.loc[info, 'coverage'].to_numpy()}), bins=3)
    return {'genes': info, 'statistic': np.sign(whole_lfc50.reindex(info).fillna(0)) * t_main, 'sets': sets_,
            'residuals': residuals.to_numpy(), 'strata': strata_}


REFERENCES = {}
for ref, spec_ in (('Census', (h_late, h_early, m_late, m_early, males)), ('Lake', (lake_late, lake_early, m_late, m_early, males))):
    REFERENCES[(ref, 'positional')] = positional_reference(*spec_)
    whole_h = h_whole if ref == 'Census' else lake_whole
    REFERENCES[(ref, 'average')] = average_reference(REFERENCES[(ref, 'positional')]['genes'], whole_h, m_whole, males)
SCORERS = {k: JointReplicationZ(r['statistic'], r['strata'], r['sets'], r['residuals']) for k, r in REFERENCES.items()}

# Guards: the scorers reproduce notebook 50's saved replication z (robust 36 positional; all average-level sets).
for ref, slug50 in (('Census', 'census_male'), ('Lake', 'lake_male')):
    saved = newest(nb50_cache, f'collection_null_{slug50}_signed_matched')
    mine = pd.Series(SCORERS[(ref, 'positional')].z(), index=SCORERS[(ref, 'positional')].ids)
    theirs = pd.Series(saved['set_z'], index=saved['set_ids'].astype(str))
    robust_here = [p for p in robust36 if p in theirs.index]
    np.testing.assert_allclose(mine[robust_here], theirs[robust_here], rtol=1e-6, atol=1e-9)
    saved = newest(nb50_cache, f'conventional_only_null_{slug50}')
    mine = pd.Series(SCORERS[(ref, 'average')].z(), index=SCORERS[(ref, 'average')].ids)
    theirs = pd.Series(saved['set_z'], index=saved['set_ids'].astype(str))
    np.testing.assert_allclose(mine[theirs.index], theirs, rtol=1e-6, atol=1e-9)
print('Guards passed: replication z equals notebook 50 for', {k: len(s.ids) for k, s in SCORERS.items()}, 'testable sets')

AVERAGE_LEVEL = {'A1', 'A2', 'A3'}
level_of = lambda key: 'average' if key.split('·')[0] in AVERAGE_LEVEL else 'positional'
spec_rows = pd.DataFrame(METHODS).set_index('key')
passed = lambda v: v is not None and bool(v)   # 'specific' holds True, False or None
ELIGIBLE = [k for k in spec_rows.index if passed(spec_rows.loc[k, 'specific']) and spec_rows.loc[k, 'species calls'] >= MIN_CALLS_EXTERNAL]
COLLECTIONS = {}
for (ref, level), scorer in SCORERS.items():
    index = {p: i for i, p in enumerate(scorer.ids)}
    COLLECTIONS[(ref, level)] = {k: np.array([index[p] for p in sorted(CALLS[k]['species']) if p in index])
                                 for k in ELIGIBLE if level_of(k) == level}


def replication_chunk(scorer, rows, strata_, seed_ids, n):
    """``n`` within-strata relabelings (seeded by the chunk's ids); mean joint z of every collection."""
    rng = np.random.default_rng(seed_ids)
    draws = np.empty((n, len(rows)))
    for b in range(n):
        z = scorer.z(stratified_permutation(strata_, rng))
        draws[b] = [z[r].mean() if len(r) else np.nan for r in rows]
    return {'draws': draws}


chunk_sizes = [len(c) for c in np.array_split(np.arange(N_PERM_EXTERNAL), EXTERNAL_CHUNKS)]
replication_tasks = {}
for run, (key_, scorer) in enumerate(SCORERS.items()):
    rows_ = list(COLLECTIONS[key_].values())
    for c, n in enumerate(chunk_sizes):
        replication_tasks[f'{key_[0]}_{key_[1]}|{c}'] = (scorer, rows_, REFERENCES[key_]['strata'], [EXTERNAL_SEED, run, c], n)
replication_runs = cached_tasks('external_replication', replication_chunk, replication_tasks,
                                params={'n_perm': N_PERM_EXTERNAL, 'chunks': EXTERNAL_CHUNKS, 'seed': EXTERNAL_SEED},
                                inputs={'references': {f'{k[0]}_{k[1]}': [digest(r['statistic']), digest(r['residuals']),
                                                                         digest(sorted(r['sets'].items()))] for k, r in REFERENCES.items()},
                                        'collections': {f'{k[0]}_{k[1]}': {m: v for m, v in c.items()} for k, c in COLLECTIONS.items()}})
replication_rows = []
for key_, scorer in SCORERS.items():
    draws = np.vstack([replication_runs[f'{key_[0]}_{key_[1]}|{c}'][0]['draws'] for c in range(EXTERNAL_CHUNKS)])
    observed_z = scorer.z()
    for j, (method, rows_) in enumerate(COLLECTIONS[key_].items()):
        observed = float(observed_z[rows_].mean()) if len(rows_) else np.nan
        replication_rows.append({'key': method, 'reference': key_[0], 'statistic': key_[1], 'calls testable': len(rows_),
                                 'mean z': observed, 'p': upper_p(observed, draws[:, j]) if len(rows_) else np.nan})
REPLICATION = pd.DataFrame(replication_rows)
REPLICATION.to_csv(tables / 'external_replication_by_method.csv', index=False)
replication_seconds = sum(sec for _, sec in replication_runs.values())
display(REPLICATION.pivot(index='key', columns='reference', values=['calls testable', 'mean z', 'p']).round(3))

#### Parallel execution check

Small subsets of the parallel tasks are run with one worker and with `N_JOBS` workers. They must
give identical outputs:
- a structure-level score GAM;
- k-means fits on resamples;
- rotation tests;
- replication chunks;
- a preranked GSEA.

In [ ]:
check_tasks = {
    'score GAM': (score_gam_task, [(SCORES['plage'][:, :25], p, *group_of(p)) for p in PARTS]),
    'k-means': (kmeans_task, [(standardize(curves['species']['delta'][top_rows(curves['species']['T_spatial'])]), 5, KM_SEED + r,
                               np.sort(np.random.default_rng(r).choice(1107, 885, replace=False))) for r in range(3)]),
    'rotation': (lambda y_, x, t: {'p': rotation_test(y_, x, t, n_rot=25)},
                 [(whole_y, *pseudobulk_designs('whole')[p]) for p in PARTS]),
    'replication': (replication_chunk, [(SCORERS[('Census', 'positional')], list(COLLECTIONS[('Census', 'positional')].values()),
                                         REFERENCES[('Census', 'positional')]['strata'], [EXTERNAL_SEED, 0, c], 3) for c in range(3)]),
    'GSEA': (gsea_task, [(gene_scores37[p]['T_spatial'], genes, gene_sets, 0) for p in PARTS[:2]])}
for name, (function, args) in check_tasks.items():
    serial, _ = fan_out(function, args, n_jobs=1)
    parallel, _ = fan_out(function, args, n_jobs=N_JOBS)
    for a, b in zip(serial, parallel):
        for k in a:
            assert np.array_equal(np.asarray(a[k]), np.asarray(b[k]), equal_nan=True), f'{name}: n_jobs changes {k}'
print(f'Parallel check passed: n_jobs = 1 and n_jobs = {N_JOBS} give identical outputs for', ', '.join(check_tasks))

### 6.2 · The summary heat-table

**Rows.** One per method, grouped by family: pathway-first (A), score-first (C), gene-first (B).
The row label gives:
- the code;
- the status: **E** existed in notebook 61, **U** is upstream (loaded and checked), **N** is new;
- "demo" for the two labelled demonstrations.

**Columns:**
- species calls, then **pass/fail for the relabeling rule** beside them, then the two relabeled
  counts and the NCDR;
- the share of tested pathways called. A share near 100% means the method does not rank pathways;
- programs: the species calls collapsed with the driver rule;
- overlap with the primary method: the Jaccard index of calls and the Spearman correlation of
  per-pathway evidence;
- external replication: the mean z of the calls, with ✱ for p ≤ 0.05 and – where it was not run;
- what the method returns:
  - **yes/no**: a call;
  - **direction**: which species is higher;
  - **location**: where along the PT;
  - **shape**: a positional profile;
  - **genes**: a gene list behind the call. A T_spatial-based A6 row returns all five, through
    the pathway's Δ(s) and its driver genes;
- the serial-equivalent compute time. This is the sum of the task times for work done here, so it
  does not depend on the number of workers; "loaded" means it was computed upstream.

Cell colour is scaled within each column.

In [ ]:
TEST_ORDER = ['ORAc', 'ORAr', 'GSEA', 'MAUC', 'CAM', 'JNT', 'SCs', 'SCu', 'markers', 'assoc', 'ends', 'GAM', 'SP',
              'T10', 'split', 'all', 'concat', 'wgcna']
SCORE_ORDER = ['meanz', 'aucell', 'ssgsea', 'plage', 'singscore']


def method_order(key):
    family = 'ACB'.index(key[0])
    parts_ = key.split('·')
    if key[0] == 'C':
        return (family, SCORE_ORDER.index(parts_[1]), TEST_ORDER.index(parts_[2]))
    return (family, int(parts_[0][1:]), TEST_ORDER.index(parts_[1]))


order = sorted(CALLS, key=method_order)
program_counts, _ = fan_out(lambda ids: {'n': np.array(driver_programs(ids).nunique())}, [(sorted(CALLS[k]['species']),) for k in order])
TABLE = pd.DataFrame(METHODS).set_index('key').loc[order]
TABLE['share called'] = TABLE['species calls'] / TABLE['tested']
TABLE['programs'] = [int(r['n']) for r in program_counts]
primary = CALLS['A6·JNT']['species']
TABLE['Jaccard with A6·JNT'] = [len(CALLS[k]['species'] & primary) / max(len(CALLS[k]['species'] | primary), 1) for k in order]
evidence = pd.DataFrame(EVIDENCE)
TABLE['Spearman with A6·JNT'] = [evidence[[k, 'A6·JNT']].dropna().corr(method='spearman').iloc[0, 1] if k in evidence else np.nan
                                 for k in order]
for ref in ('Census', 'Lake'):
    block = REPLICATION[REPLICATION.reference.eq(ref)].set_index('key')
    TABLE[f'{ref} z'] = block['mean z'].reindex(order)
    TABLE[f'{ref} p'] = block['p'].reindex(order)
TABLE.to_csv(tables / 'methods.csv')
display(TABLE[['method', 'status', 'species calls', 'specific', 'relabeled calls', 'NCDR', 'share called', 'programs',
               'Jaccard with A6·JNT', 'Spearman with A6·JNT', 'Census z', 'Census p', 'Lake z', 'Lake p', 'runtime (s)']].round(3))


def heat_table(table):
    columns = [('species calls', 'species\ncalls'), ('specific', 'relabel\nrule'), ('relabeled calls', 'relabeled\ncalls'),
               ('NCDR', 'NCDR'), ('share called', '%\ntested'), ('programs', 'pro-\ngrams'),
               ('Jaccard with A6·JNT', 'Jaccard\nwith\nA6·JNT'), ('Spearman with A6·JNT', 'Spear-\nman with\nA6·JNT'),
               ('Census z', 'Census\nrepl. z'), ('Lake z', 'Lake\nrepl. z'),
               *[(r, r.replace('/', '/\n')) for r in RETURNS], ('runtime (s)', 'serial\nseconds')]
    n = len(table)
    height = 21 + 3.3 * n
    fig = plt.figure(figsize=(180 * MM, height * MM))
    ax = fig.add_axes([.37, .005, .625, (3.3 * n) / height])
    ax.set_xlim(0, len(columns))
    ax.set_ylim(n, 0)
    ax.axis('off')
    cmap = plt.get_cmap('Greys')
    for j, (column, title) in enumerate(columns):
        ax.text(j + .5, -.3, title, ha='center', va='bottom', fontsize=4.6, linespacing=1.05)
        values = table[column]
        numeric = pd.to_numeric(values, errors='coerce') if column not in ('specific', 'relabeled calls') else None
        for i, (key, v) in enumerate(values.items()):
            face, text, color = 'white', '', INK
            if column == 'specific':
                face = {True: '#CFE8D6', False: '#F4CCCC'}.get(v, '#F2F2F2')
                text = {True: 'pass', False: 'fail'}.get(v, 'n/a')
            elif column == 'relabeled calls':
                text = str(v)
            elif column in RETURNS:
                text = '✓' if v else ''
            elif column == 'runtime (s)':
                text = 'loaded' if v == LOADED else ('<1' if float(v) < 1 else f'{float(v):.0f}') if v is not None else ''
            elif pd.notna(numeric[key]):
                x = float(numeric[key])
                scale = numeric.abs().max() if column not in ('share called', 'NCDR') else 1.
                level = min(abs(x) / scale, 1) if scale else 0
                if column in ('Census z', 'Lake z'):
                    face = plt.get_cmap('PuOr_r')(.5 + .5 * np.clip(x / 6, -1, 1))
                    p_ = table.loc[key, column.replace(' z', ' p')]
                    text = f'{x:.1f}' + ('✱' if p_ <= .05 else '')
                else:
                    face = cmap(.08 + .55 * (np.log1p(level * 50) / np.log1p(50) if column == 'species calls' else level))
                    text = (f'{x:.0%}' if column == 'share called' else f'{x:.2f}' if column in ('NCDR', 'Jaccard with A6·JNT', 'Spearman with A6·JNT')
                            else f'{x:.0f}')
                color = 'white' if (column not in ('Census z', 'Lake z') and level > .7) else INK
            elif column in ('Census z', 'Lake z'):
                text = '–'
            ax.add_patch(plt.Rectangle((j + .04, i + .06), .92, .88, facecolor=face, edgecolor='none'))
            ax.text(j + .5, i + .5, text, ha='center', va='center', fontsize=4.6, color=color)
    for i, key in enumerate(table.index):
        row = table.loc[key]
        tag = row['status'] + (' demo' if '[demonstration]' in row['method'] else '')
        label = row['method'].replace(' [demonstration]', '')
        ax.text(-.15, i + .5, f'{key}  [{tag}]  {label[:46]}', fontsize=4.8, va='center', ha='right', clip_on=False,
                color=FAMILY_COLORS[row['family']])
    previous = None
    for i, key in enumerate(table.index):
        if previous is not None and table.loc[key, 'family'] != previous:
            ax.axhline(i, color=INK, lw=.6)
        previous = table.loc[key, 'family']
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAMILY_COLORS.values()]
    fig.legend(handles, list(FAMILY_COLORS) + [], loc='upper left', ncol=3, fontsize=5, bbox_to_anchor=(.005, 1))
    fig.text(.005, 1 - 7 / height, '[E] existed in notebook 61 · [U] upstream, loaded and checked · [N] new · demo = labelled '
             'demonstration · ✱ replication p ≤ 0.05 · – not run (not specific or < 5 calls)', fontsize=4.6, color=MUTED)
    return fig


fig = heat_table(TABLE)
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig0_method_heat_table.{ext}', bbox_inches='tight')
plt.show()

### 6.3 · Overlap of species calls

**a.** The Jaccard index between the species call sets of every pair of methods with calls.

**b.** An UpSet view by family. Each family is the union of its specific methods' calls, with the
primary method shown on its own.

A deviation from the frozen plan is reported here. Specific methods that call more than half of
the tested pathways are left out of the unions:
- the self-contained rotation tests;
- the score split plots.

Including them makes every family union nearly all 1,513 pathways, and the intersections then show
nothing. The frozen-rule intersections are saved in `upset_family_frozen_rule.csv`.

In [ ]:
keys = [k for k in order if k != 'A0·markers' and len(CALLS[k]['species'])]
jac = pd.DataFrame([[len(CALLS[a]['species'] & CALLS[b]['species']) / max(len(CALLS[a]['species'] | CALLS[b]['species']), 1)
                     for b in keys] for a in keys], index=keys, columns=keys)
jac.to_csv(tables / 'jaccard_species_calls.csv')
specific_keys = [k for k in order if passed(TABLE.loc[k, 'specific'])]
selective = [k for k in specific_keys if TABLE.loc[k, 'share called'] <= .5]
FAMILY_UNION = {f: set().union(*[CALLS[k]['species'] for k in selective if TABLE.loc[k, 'family'] == f]) for f in FAMILY_COLORS}
FAMILY_UNION['A6·JNT (primary)'] = set(primary)
frozen_union = {f: set().union(*[CALLS[k]['species'] for k in specific_keys if TABLE.loc[k, 'family'] == f]) for f in FAMILY_COLORS}
pd.DataFrame({f: [p in s for p in pathways] for f, s in frozen_union.items()}, index=pathways).pipe(
    lambda m: m[m.any(axis=1)].value_counts()).rename('pathways').to_frame().to_csv(tables / 'upset_family_frozen_rule.csv')
print('Left out of the family unions (specific, but calling > 50% of tested):', [k for k in specific_keys if k not in selective])
print('Family unions (selective specific methods):', {f: len(s) for f, s in FAMILY_UNION.items()})
upset_rows = pd.DataFrame({f: [p in s for p in pathways] for f, s in FAMILY_UNION.items()}, index=pathways)
combos = upset_rows[upset_rows.any(axis=1)].value_counts().sort_values(ascending=False)
combos.rename('pathways').to_frame().to_csv(tables / 'upset_family_intersections.csv')

fig = plt.figure(figsize=(180 * MM, 100 * MM))
gs = fig.add_gridspec(2, 2, width_ratios=[1.25, 1], height_ratios=[1.4, 1], wspace=.42, hspace=.08)
ax = fig.add_subplot(gs[:, 0])
image = ax.imshow(jac.to_numpy(), cmap='Greens', vmin=0, vmax=1)
ax.set_xticks(range(len(keys)), keys, rotation=90, fontsize=3.8)
ax.set_yticks(range(len(keys)), keys, fontsize=3.8)
for tick, key in zip(ax.get_yticklabels(), keys):
    tick.set_color(FAMILY_COLORS[TABLE.loc[key, 'family']])
for tick, key in zip(ax.get_xticklabels(), keys):
    tick.set_color(FAMILY_COLORS[TABLE.loc[key, 'family']])
ax.set_title('a  Jaccard index of species calls', loc='left')
fig.colorbar(image, ax=ax, fraction=.03, pad=.01, shrink=.6).ax.tick_params(labelsize=5)
names_upset = list(FAMILY_UNION)
top_combos = combos.head(12)
bars = fig.add_subplot(gs[0, 1])
dots = fig.add_subplot(gs[1, 1], sharex=bars)
x = np.arange(len(top_combos))
bars.bar(x, top_combos.to_numpy(), color=INK, width=.6)
for xi, v in zip(x, top_combos.to_numpy()):
    bars.text(xi, v, str(v), ha='center', va='bottom', fontsize=5)
bars.set_ylabel('pathways')
bars.set_title('b  Intersections of family unions (specific, selective methods)', loc='left')
plt.setp(bars.get_xticklabels(), visible=False)
for i, f in enumerate(names_upset):
    present = np.array([combo[i] for combo in top_combos.index])
    dots.scatter(x, np.full(len(x), i), s=12, color=np.where(present, FAMILY_COLORS.get(f, INK), LIGHT), zorder=3)
for xi, combo in zip(x, top_combos.index):
    on = [i for i, flag in enumerate(combo) if flag]
    if len(on) > 1:
        dots.plot([xi, xi], [min(on), max(on)], color=INK, lw=1)
dots.set_yticks(range(len(names_upset)), [f'{f.split(" · ")[0] if " · " in f else f} ({len(FAMILY_UNION[f])})' for f in names_upset])
dots.set_xticks([])
dots.invert_yaxis()
for side in ('left', 'bottom'):
    dots.spines[side].set_visible(False)
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig1_overlap.{ext}', bbox_inches='tight')
plt.show()

### 6.4 · Direction agreement among the signed methods

The signed methods, and how each gives a direction:
- **A1·GSEA:** the sign of NES;
- **A1·ORAc** and **A3·ORAc:** the DE list (up or down) with the strongest enrichment;
- **A3·GSEA:** the sign in the segment with the strongest test;
- **A6·JNT:** the sign of the member-mean Δ(s) at its largest absolute value (human − mouse).

Concordance is the share of pathways called by both methods with the same sign. Rank agreement
with the primary method is a column of the heat-table above.

In [ ]:
member_index = {p: pd.Index(genes).get_indexer(v) for p, v in gene_sets.items()}
pathway_delta = pd.DataFrame({p: curves['species']['delta'][idx].mean(axis=0) for p, idx in member_index.items()},
                             index=np.round(grid, 4)).T
peak = pathway_delta.abs().idxmax(axis=1)
DIRECTION['A6·JNT'] = pd.Series(np.sign([pathway_delta.loc[p, peak[p]] for p in pathways]), index=pathways)
signed = ['A1·GSEA', 'A1·ORAc', 'A3·GSEA', 'A3·ORAc', 'A6·JNT']
concordance = pd.DataFrame(index=signed, columns=signed, dtype=float)
shared = pd.DataFrame(index=signed, columns=signed, dtype=float)
for a in signed:
    for b in signed:
        both = sorted(CALLS[a]['species'] & CALLS[b]['species'])
        shared.loc[a, b] = len(both)
        concordance.loc[a, b] = (DIRECTION[a][both] == DIRECTION[b][both]).mean() if both else np.nan
concordance.to_csv(tables / 'direction_concordance.csv')
shared.to_csv(tables / 'direction_concordance_n.csv')
evidence.corr(method='spearman').to_csv(tables / 'rank_correlation_of_evidence.csv')
print('Direction concordance among pathways called by both (n in the second table):')
display(concordance.round(2))
display(shared.astype(int))

### 6.5 · What the discrete methods miss, and what they add

The specific whole-PT and discrete methods are A1·ORAc, A3·ORAc and A4·JNT. For the primary calls
they miss, notebook 50's step-screen q shows how close the discrete screen came.

In [ ]:
specific_wd = CALLS['A1·ORAc']['species'] | CALLS['A3·ORAc']['species'] | CALLS['A4·JNT']['species']
C = CALLS['A6·JNT']['species']
misses = pd.Series({
    'A6·JNT calls': len(C),
    'A6·JNT not called by A1·ORAc, A3·ORAc or A4·JNT': len(C - specific_wd),
    'A6·JNT not called by the segment-step screen (A4·JNT)': len(C - CALLS['A4·JNT']['species']),
    'A4·JNT calls not called by A6·JNT': len(CALLS['A4·JNT']['species'] - C),
    'A3·ORAc calls not called by A6·JNT': len(CALLS['A3·ORAc']['species'] - C),
    'A1·ORAc calls not called by A6·JNT': len(CALLS['A1·ORAc']['species'] - C)}, name='pathways')
misses.to_csv(tables / 'misses.csv')
display(misses.to_frame())
only_c = sorted(C - CALLS['A4·JNT']['species'])
detail = pd.DataFrame({'name': NAMES[only_c], 'A6·JNT z_joint': EVIDENCE['A6·JNT'][only_c],
                       'step-screen q (A4·JNT)': steps50.q_joint.reindex(only_c),
                       'called by A3·ORAc': [p in CALLS['A3·ORAc']['species'] for p in only_c]}).sort_values('A6·JNT z_joint', ascending=False)
detail.to_csv(tables / 'continuous_not_steps.csv')
display(detail.round(3))

### 6.6 · Gene-first against pathway-first: do the modules find the primary programs?

For each gene-first method, two measures against the primary method's programs (section 6.0):
- **Recovered.** A program counts as recovered when at least one of its pathways is among the
  method's species calls.
- **Best Jaccard.** The best Jaccard index between the program's driver genes and any
  annotated or called module's genes, shown as the median over programs.

The reverse is also counted: modules with no enriched pathway, which no library set describes.

In [ ]:


def driver_genes(ids):
    union = set()
    for p in ids:
        members = set(gene_sets[p])
        chosen = members & contributing
        union |= chosen if len(chosen) >= 3 else members
    return union


program_members = {prog: sorted(block.index) for prog, block in c1_drivers.groupby(c1_drivers)}
recovery_rows = []
for key in [k for k in order if k.startswith('B')]:
    modules = [set(m) for m in MODULE_GENES.get(key, [])]
    best = [max((len(driver_genes(ids) & m) / len(driver_genes(ids) | m) for m in modules), default=0.)
            for ids in program_members.values()]
    recovered = sum(any(p in CALLS[key]['species'] for p in ids) for ids in program_members.values())
    recovery_rows.append({'method': key, 'specific': TABLE.loc[key, 'specific'], 'species calls': len(CALLS[key]['species']),
                          'primary programs recovered': f'{recovered} of {len(program_members)}',
                          'median best Jaccard (driver genes vs module)': float(np.median(best)),
                          'programs with best Jaccard ≥ 0.3': int(np.sum(np.array(best) >= .3))})
RECOVERY = pd.DataFrame(recovery_rows).set_index('method')
unannotated = {key: int((table_.get('enriched pathways', pd.Series(dtype=int)) == 0).sum()) for key, table_ in MODULE_TABLES.items()}
unannotated['B5·wgcna'] = int((B5_TABLE['enriched pathways'] == 0).sum())
RECOVERY['species modules without any enriched pathway'] = pd.Series(unannotated)
RECOVERY.to_csv(tables / 'gene_first_recovery_of_primary_programs.csv')
display(RECOVERY.round(3))

## 7 · Worked examples: one pathway score, three views

Every view shows the same per-structure score, the mean-z module score of method C·meanz·SP:
- **whole PT:** specimen means;
- **S1/S2/S3:** specimen × segment means, bars for the species mean;
- **continuous:** a specimen-balanced species curve (6-df spline) with specimen × 12-bin means as
  points.

The pathways were chosen by the rules fixed in `plan.md` before the notebook ran:

In [ ]:
beyond = pd.read_csv(nb57 / 'per_pathway_beyond_steps.csv', index_col=0)
ranks = evidence.rank(pct=True)
rules = {}
agree = sorted(CALLS['A1·ORAc']['species'] & CALLS['A3·ORAc']['species'] & CALLS['A4·JNT']['species'] & C)
rules['E1 · all methods agree'] = ranks.loc[agree, ['A1·ORAc', 'A3·ORAc', 'A4·JNT', 'A6·JNT']].min(axis=1).sort_values(ascending=False).index[:1] if agree else []
flat = [p for p in C if steps50.q_joint.get(p, 1) > .05 and p not in CALLS['A3·GSEA']['species'] and p not in CALLS['A3·ORAc']['species']
        and bool(beyond.loc[p, 'called test_B_coordinate_steps'])]
rules['E2 · segment means flat, curves structured'] = beyond.loc[flat, 'z_joint test_B_coordinate_steps'].sort_values(ascending=False).index[:1] if flat else []
discrete_only = sorted(CALLS['A3·ORAc']['species'] - C)
rules['E3 · discrete call, no continuous call'] = EVIDENCE['A3·ORAc'][discrete_only].sort_values(ascending=False).index[:1] if discrete_only else []
reversing = []
for p in C:
    d = pathway_delta.loc[p].to_numpy()
    threshold = .25 * np.abs(d).max()
    if (d >= threshold).any() and (d <= -threshold).any():
        reversing.append(p)
rules['E4 · direction depends on position'] = EVIDENCE['A6·JNT'][reversing].sort_values(ascending=False).index[:1] if reversing else []
steplike = [p for p in C & CALLS['A4·JNT']['species'] if not bool(beyond.loc[p, 'called test_B_coordinate_steps'])]
rules['E5 · step-like: continuous adds nothing'] = steps50.z_joint[steplike].sort_values(ascending=False).index[:1] if steplike else []
candidates = {'E1': len(agree), 'E2': len(flat), 'E3': len(discrete_only), 'E4': len(reversing), 'E5': len(steplike)}
EXAMPLES = {}
for rule, picked in rules.items():
    if len(picked):
        EXAMPLES[rule] = picked[0]
        print(f'{rule}: {NAMES[picked[0]]} (pathways meeting the rule: {candidates[rule[:2]]})')
    else:
        print(f'{rule}: no pathway meets the rule')

In [ ]:
score_matrix = scores['mean_z']
weights = np.zeros(len(position))
for sp, names_ in SPECIMENS.items():
    for n in names_:
        rows = specimen == n
        weights[rows] = len(position) / (2 * len(names_) * rows.sum())
fine_bins = np.clip(((position - lo) / (hi - lo) * EXAMPLE_BINS).astype(int), 0, EXAMPLE_BINS - 1)
transitions = pd.read_csv(nb37 / 'tables' / 'segment_transitions_by_specimen.csv').groupby('transition').position.mean()


def species_curve(values, species_mask):
    design = make_gam_design(position[species_mask], knots)
    w = np.sqrt(weights[species_mask])
    beta, *_ = np.linalg.lstsq(w[:, None] * design, w * values[species_mask], rcond=None)
    return make_gam_design(grid, knots) @ beta


fig, axes = plt.subplots(len(EXAMPLES), 3, figsize=(180 * MM, 38 * MM * len(EXAMPLES)),
                         gridspec_kw={'width_ratios': [.55, 1, 1.8]}, squeeze=False)
example_rows = []
for r, (rule, pathway) in enumerate(EXAMPLES.items()):
    values = score_matrix[:, pathways.index(pathway)]
    ax_w, ax_s, ax_c = axes[r]
    for k, (sp, color) in enumerate((('mouse', MOUSE), ('human', HUMAN))):
        means = [values[specimen == n].mean() for n in SPECIMENS[sp]]
        ax_w.bar(k, np.mean(means), color=color, alpha=.35, width=.6)
        for n, m in zip(SPECIMENS[sp], means):
            ax_w.scatter(k, m, color=color, marker=SPECIMEN_MARKER[n], s=10, zorder=3, edgecolor='white', linewidth=.4)
        for j, seg in enumerate(('PT-S1', 'PT-S2', 'PT-S3')):
            seg_means = [values[(specimen == n) & (segment == seg)].mean() for n in SPECIMENS[sp]]
            xpos = j + (k - .5) * .38
            ax_s.bar(xpos, np.mean(seg_means), width=.36, color=color, alpha=.35)
            for n, m in zip(SPECIMENS[sp], seg_means):
                ax_s.scatter(xpos, m, color=color, marker=SPECIMEN_MARKER[n], s=10, zorder=3, edgecolor='white', linewidth=.4)
        mask = ~human if sp == 'mouse' else human
        ax_c.plot(grid, species_curve(values, mask), color=color, lw=1.4)
        for n in SPECIMENS[sp]:
            rows = specimen == n
            b = pd.Series(values[rows]).groupby(fine_bins[rows]).mean()
            centres = lo + (b.index.to_numpy() + .5) * (hi - lo) / EXAMPLE_BINS
            ax_c.scatter(centres, b.to_numpy(), color=color, marker=SPECIMEN_MARKER[n], s=7, alpha=.8,
                         edgecolor='white', linewidth=.3, zorder=3)
    for t in transitions:
        ax_c.axvline(t, color=NULL, lw=.6, ls=':')
    ax_w.set_xticks([0, 1], ['mouse', 'human'])
    ax_s.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
    ax_c.set_xlim(lo, hi)
    ax_w.set_ylabel('pathway score (mean z)')
    calls_here = [k for k in ('A1·GSEA', 'A1·ORAc', 'A3·GSEA', 'A3·ORAc', 'A4·JNT', 'A6·JNT', 'C·meanz·SP') if pathway in CALLS[k]['species']]
    ax_w.set_title(f'{rule.split(" · ")[0]}  whole PT', loc='left')
    ax_s.set_title('S1 / S2 / S3', loc='left')
    ax_c.set_title(f'{NAMES[pathway][:60]} · called by {", ".join(calls_here) or "none"}', loc='left')
    if r == len(EXAMPLES) - 1:
        ax_c.set_xlabel('PT coordinate (SCF13); dotted lines: mean S1→S2 and S2→S3 transitions')
    example_rows.append({'example': rule, 'pathway': NAMES[pathway], 'called by': ', '.join(calls_here),
                         'step-screen q (A4·JNT)': steps50.q_joint.get(pathway, np.nan),
                         'A6·JNT q_joint': tests37[tests37.statistic.eq('T_spatial') & tests37.partition.eq('species')
                                               & tests37.pathway_id.eq(pathway)].q_joint.iloc[0],
                         'beyond coordinate steps called (notebook 57)': bool(beyond.loc[pathway, 'called test_B_coordinate_steps'])})
handles = [plt.Line2D([], [], color=MOUSE, lw=1.4, label='mouse'), plt.Line2D([], [], color=HUMAN, lw=1.4, label='human'),
           plt.Line2D([], [], color=INK, marker='o', lw=0, markersize=3, label='specimen 1'),
           plt.Line2D([], [], color=INK, marker='s', lw=0, markersize=3, label='specimen 2')]
axes[0, 2].legend(handles=handles, loc='upper left', ncol=4, bbox_to_anchor=(0, 1.32))
fig.tight_layout()
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig2_worked_examples.{ext}', bbox_inches='tight')
plt.show()
EXAMPLE_TABLE = pd.DataFrame(example_rows)
EXAMPLE_TABLE.to_csv(tables / 'worked_examples.csv', index=False)
display(EXAMPLE_TABLE.round(4))

## 8 · Gene-level heatmaps by program: which genes drive the signal, and where

A pathway score averages its members, so it hides the genes that drive the call and cancels members
that change in opposite directions. These heatmaps show the genes themselves, one heatmap per program
(section 6.0), so near-duplicate pathways are not shown twice.

- **Rows.** The union of the member pathways' contributing genes (top 10% by T_spatial; all members
  when fewer than three qualify), at most 40, chosen by T_spatial. A bullet marks top-decile genes.
  The left matrix shows which member pathway, numbered in the title, contains each gene.
- **Columns.** The 12 coordinate bins. Mouse and human sit side by side. Each gene's
  specimen-averaged bin means are z-scored within each species, which removes the level offset.
  Dotted lines mark each species' mean S1→S2 and S2→S3 transitions; a dot marks the peak bin.
- **Row order.** By mouse peak bin, then human peak bin, so waves of genes switching on or off are
  visible.
- **Right of the heatmap:**
  - notebook 43 zonation class;
  - log10 T_spatial;
  - whole-PT DESeq2 log2 human/mouse (H/M), so the offset is not lost.
- **Below:** each specimen's own z-scored bins in the same order, the replication check.

The notebook shows the programs that contain the worked examples, among the 60 primary-test
(A6·JNT) calls. E3 is not a primary-test call and is shown alone. The gallery `figures/gene_heatmaps_primary22.pdf` has one page per
program of the 22 primary pathways. Display rules were fixed in `plan.md` before this section ran.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LinearSegmentedColormap

MAX_ROWS, Z_CLIP = 40, 2.5
HEAT = LinearSegmentedColormap.from_list('teal_grey_brown', ['#01665E', '#5AB4AC', '#E6E6E6', '#D8B365', '#8C510A'])
CLASS_COLORS = {'conserved': '#009E73', 'reversal': '#CC79A7', 'mouse-only': MOUSE, 'human-only': HUMAN,
                'indeterminate': '#E3E3E3', 'neither': '#B8B8B8'}
SHORT_SPECIMEN = {'Ctrl1A2': 'M1', 'Ctrl1A4': 'M2', 'HUK1_COR1': 'H1', 'HUK1_MED1': 'H2'}


def specimen_bin_means():
    """Mean log-normalised expression per specimen x coordinate bin (genes x bins per specimen)."""
    out = {}
    for n in ALL_SPECIMENS:
        rows = np.flatnonzero(specimen == n)
        indicator = sparse.csr_matrix((np.ones(len(rows)), (fine_bins[rows], np.arange(len(rows)))),
                                      shape=(EXAMPLE_BINS, len(rows)))
        sums = np.asarray((indicator @ inputs['Y'][rows]).todense())
        out[n] = (sums / np.asarray(indicator.sum(axis=1))).T
    return out


bin_means = cached_payload('specimen_bin_means', specimen_bin_means, root=cache, params={'bins': EXAMPLE_BINS},
                           inputs=input_key, code=code)
species_means = {sp: np.mean([bin_means[n] for n in SPECIMENS[sp]], axis=0) for sp in SPECIMENS}


def zscore_rows(values):
    centred = values - values.mean(axis=1, keepdims=True)
    sd = values.std(axis=1, keepdims=True)
    return np.where(sd > 1e-9, centred / np.where(sd > 1e-9, sd, 1), 0.)


species_z = {sp: zscore_rows(v) for sp, v in species_means.items()}
specimen_z = {n: zscore_rows(v) for n, v in bin_means.items()}
gene_position = pd.Series(np.arange(len(genes)), index=genes)
t_spatial = gene37
top_decile = set(t_spatial.nlargest(int(round(TOP_SHARE * len(genes)))).index)
whole_lfc = deseq[deseq.partition.eq('species') & deseq.scope.eq('whole_PT')].set_index('gene').log2FoldChange
zonation = pd.read_csv(results_root / 'pt_revision_classes' / 'reviewed' / 'tables' / 'gene_classes_symmetric.csv',
                       index_col=0)['union S2-S1 + S3c-early']
species_transitions = pd.read_csv(nb37 / 'tables' / 'segment_transitions_by_specimen.csv').groupby(
    ['species', 'transition']).position.mean()
bin_edges = np.linspace(lo, hi, EXAMPLE_BINS + 1)


def program_rows(member_pathways):
    """Union of the members' contributing genes (fallback: all members), capped at the top 40 by T_spatial."""
    union = []
    for p in member_pathways:
        members = [g for g in gene_sets[p] if g in gene_position.index]
        chosen = [g for g in members if g in contributing]
        union.extend(chosen if len(chosen) >= 3 else members)
    union = sorted(set(union))
    shown = union if len(union) <= MAX_ROWS else list(t_spatial[union].nlargest(MAX_ROWS).index)
    idx = gene_position[shown].to_numpy()
    peaks = pd.DataFrame({'mouse': species_z['mouse'][idx].argmax(axis=1), 'human': species_z['human'][idx].argmax(axis=1),
                          'gene': shown})
    order = peaks.sort_values(['mouse', 'human', 'gene']).index.to_numpy()
    return [shown[i] for i in order], len(union)


def draw_program_heatmap(member_pathways, title, rows=None):
    """Gene heatmap of a program (rows from its members' driver genes) or of given genes (``rows``, e.g. a module)."""
    member_pathways = sorted(member_pathways, key=lambda p: -EVIDENCE['A6·JNT'].get(p, 0))
    if rows is None:
        rows, n_union = program_rows(member_pathways)
    else:
        n_union = len(rows)
    idx = gene_position[rows].to_numpy()
    n, k = len(rows), len(member_pathways)
    height_mm = 40 + n * 2.4 * 1.6 + 3 * k
    fig = plt.figure(figsize=(180 * MM, height_mm * MM))
    widths = [max(k, 1) * .9, 12, 1.2, 12, 1.2, 2.4, 2.4]
    outer = fig.add_gridspec(2, 1, height_ratios=[1, .6], hspace=.16, top=1 - (14 + 3 * k) / height_mm, bottom=.04,
                             left=.12, right=.98)
    top = outer[0].subgridspec(1, 7, width_ratios=widths, wspace=.06)
    bottom = outer[1].subgridspec(1, 7, width_ratios=widths, wspace=.06)
    mx = fig.add_subplot(top[0, 0])
    matrix = np.array([[g in set(gene_sets[p]) for p in member_pathways] for g in rows], float)
    yy, xx = np.nonzero(matrix)
    mx.scatter(xx, yy, s=4, color=INK, linewidths=0)
    mx.set_xlim(-.5, max(k, 1) - .5)
    mx.set_ylim(n - .5, -.5)
    mx.set_xticks(range(k), [str(i + 1) for i in range(k)], fontsize=5)
    mx.xaxis.tick_top()
    mx.set_yticks(range(n), [('• ' if g in top_decile else '') + g for g in rows], fontsize=5)
    for side in ('right', 'bottom'):
        mx.spines[side].set_visible(False)
    mx.set_title('in pathway' if k else '', loc='left', fontsize=5.5, pad=8)
    for col, sp in ((1, 'mouse'), (3, 'human')):
        ax = fig.add_subplot(top[0, col])
        image = ax.imshow(np.clip(species_z[sp][idx], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP,
                          aspect='auto', interpolation='nearest')
        ax.scatter(species_z[sp][idx].argmax(axis=1), np.arange(n), s=3, color=INK, zorder=3, linewidths=0)
        for t in ('S1→S2', 'S2→S3'):
            ax.axvline((species_transitions[(sp, t)] - lo) / (hi - lo) * EXAMPLE_BINS - .5, color=INK, lw=.6, ls=':')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'{sp}: specimen-averaged, z within species', loc='left', color=MOUSE if sp == 'mouse' else HUMAN)
        sub = bottom[0, col].subgridspec(1, 2, wspace=.08)
        for j, name in enumerate(SPECIMENS[sp]):
            bx = fig.add_subplot(sub[0, j])
            bx.imshow(np.clip(specimen_z[name][idx], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP,
                      aspect='auto', interpolation='nearest')
            for t in ('S1→S2', 'S2→S3'):
                bx.axvline((species_transitions[(sp, t)] - lo) / (hi - lo) * EXAMPLE_BINS - .5, color=INK, lw=.5, ls=':')
            bx.set_xticks([0, EXAMPLE_BINS - 1], ['start', 'end'], fontsize=5)
            bx.set_yticks([])
            bx.set_title(f'{SHORT_SPECIMEN[name]} ({name})', loc='left', fontsize=5.5, fontweight='normal')
    cx = fig.add_subplot(top[0, 4])
    for i, g in enumerate(rows):
        c = zonation.get(g, np.nan)
        cx.add_patch(plt.Rectangle((0, i - .4), 1, .8, color=CLASS_COLORS.get(c, '#FFFFFF'),
                                   ec=LIGHT if c not in CLASS_COLORS else 'none', lw=.3))
    cx.set_xlim(0, 1)
    cx.set_ylim(n - .5, -.5)
    cx.axis('off')
    cx.set_title('class', loc='left', fontsize=5.5)
    tx = fig.add_subplot(top[0, 5])
    tx.barh(np.arange(n), np.log10(t_spatial[rows].clip(lower=1)), color=ACCENT, height=.7)
    tx.set_ylim(n - .5, -.5)
    tx.set_yticks([])
    tx.set_title('log10 T', loc='left', fontsize=5.5)
    tx.tick_params(labelsize=5)
    lx = fig.add_subplot(top[0, 6])
    lfc = whole_lfc.reindex(rows).clip(-6, 6).fillna(0).to_numpy()
    lx.barh(np.arange(n), lfc, color=np.where(lfc > 0, HUMAN, MOUSE), height=.7)
    lx.axvline(0, color=INK, lw=.5)
    lx.set_xlim(-6.5, 6.5)
    lx.set_ylim(n - .5, -.5)
    lx.set_yticks([])
    lx.set_title('log2 H/M', loc='left', fontsize=5.5)
    lx.tick_params(labelsize=5)
    cax = fig.add_subplot(bottom[0, 4:])
    cax.axis('off')
    bar = fig.colorbar(image, ax=cax, orientation='horizontal', fraction=.45, aspect=10, pad=0)
    bar.set_label('z within species (clipped ±2.5)', fontsize=5)
    bar.ax.tick_params(labelsize=5)
    legend = [plt.Rectangle((0, 0), 1, 1, color=v) for v in CLASS_COLORS.values()] + \
        [plt.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor=LIGHT)]
    cax.legend(legend, list(CLASS_COLORS) + ['not classified'], loc='upper left', fontsize=4.5, ncol=2,
               handlelength=1, borderaxespad=0)
    called = [m for m in ('A1·GSEA', 'A1·ORAc', 'A3·GSEA', 'A3·ORAc', 'A4·JNT', 'A6·JNT') if any(p in CALLS[m]['species'] for p in member_pathways)]
    header = f'{title} · {n} of {n_union} genes shown · members called by {", ".join(called) or "none"}'
    listing = '; '.join(f'{i + 1} {NAMES[p]}' for i, p in enumerate(member_pathways))
    fig.text(.01, 1 - 4 / height_mm, header, ha='left', va='top', fontsize=6.5, fontweight='bold')
    fig.text(.01, 1 - 8 / height_mm, listing, ha='left', va='top', fontsize=5, wrap=True, color=MUTED)
    return fig


def program_of(pathway, labels):
    return sorted(labels.index[labels.eq(labels[pathway])]) if pathway in labels.index else [pathway]


c1_names = program_names(c1_drivers, EVIDENCE['A6·JNT'])
for rule, pathway in EXAMPLES.items():
    members = program_of(pathway, c1_drivers)
    name = c1_names[pathway] if pathway in c1_names.index else NAMES[pathway]
    kind = f'program "{name}" ({len(members)} pathways among the primary-test calls)' if pathway in c1_drivers.index else 'one pathway (not a primary-test call)'
    fig = draw_program_heatmap(members, f'{rule.split(" · ")[0]} · {kind}')
    for ext in ('pdf', 'png'):
        fig.savefig(figures / f'fig3_gene_heatmap_{rule.split(" · ")[0]}.{ext}')
    plt.show()

primary_names = program_names(primary_drivers, EVIDENCE['A6·JNT'])
gallery = []
with PdfPages(figures / 'gene_heatmaps_primary22.pdf') as pdf:
    for program, block in primary_drivers.groupby(primary_drivers):
        members = sorted(block.index)
        fig = draw_program_heatmap(members, f'Program {program}: {primary_names[members[0]]} ({len(members)} of the primary 22)')
        pdf.savefig(fig)
        plt.close(fig)
        gallery.append({'program': program, 'name': primary_names[members[0]], 'pathways': len(members),
                        'members': '; '.join(NAMES[members])})
GALLERY = pd.DataFrame(gallery)
GALLERY.to_csv(tables / 'primary22_program_gallery.csv', index=False)
display(GALLERY)
print(f'Gallery: {len(GALLERY)} programs of the primary 22, one page each, in figures/gene_heatmaps_primary22.pdf')

### A gene-first module that the pathway-first programs do not show

The rule, fixed in `plan_expansion.md` before the run: a B5 module is drawn when it is called for
species, is called in neither relabeling, and has no A6·JNT call among its enriched pathways.
- **Rows:** its 40 highest-T_spatial genes, ordered by peak bin as above.
- **Left matrix:** its (at most five) most enriched pathways.

In [ ]:
shown_modules = [m for m in range(n_modules) if module_called['species'][m] and not any(module_called[s][m] for s in SWAPS)
                 and B5_TABLE.loc[m, 'A6·JNT calls among them'] == 0]
print('B5 modules meeting the heatmap rule:', [m + 1 for m in shown_modules] or 'none')
for m in shown_modules:
    module_genes = list(np.array(b5_universe)[b5['labels'] == m])
    top_genes = list(t_spatial[module_genes].nlargest(MAX_ROWS).index)
    idx = gene_position[top_genes].to_numpy()
    peaks = pd.DataFrame({'mouse': species_z['mouse'][idx].argmax(axis=1), 'human': species_z['human'][idx].argmax(axis=1),
                          'gene': top_genes}).sort_values(['mouse', 'human', 'gene'])
    enriched = [p for p in np.array(pathways)[np.argsort(b5_ora[:, m])[:5]] if b5_q[pathways.index(p), m] <= ALPHA]
    fig = draw_program_heatmap(enriched, f'B5 module {m + 1}: {len(module_genes)} genes, eigengene q = '
                               f'{B5_TABLE.loc[m, "q species"]:.2g}; top 40 by T_spatial', rows=list(peaks.gene))
    for ext in ('pdf', 'png'):
        fig.savefig(figures / f'fig4_b5_module_{m + 1}.{ext}')
    plt.show()

## 9 · Which method for which question

Every number below comes from the run above. Calls are given as species calls (relabeled calls);
replication is the mean external z of the calls, Census / Lake, with p from the overlap-preserving
null.

| Question | Recommended method | Why | Where the continuous coordinate adds information |
|---|---|---|---|
| Does a pathway differ **on average** over the whole PT? | **A1·ORAc**: whole-PT DESeq2, ORA on the DE lists | Specific (80; 0, 1); replicated in Lake (z 0.79, p 0.011) but not Census (0.56, p 0.071); about 60% of calls are probe-exposed (notebook 56) | None. No coordinate method tests an offset (E3, oxidative phosphorylation) |
| Does it differ **within a segment**? | **A3·ORAc**: S1/S2/S3 DESeq2 + ORA | Specific (118; 0, 1); gives segment and direction; Lake 0.60 (p 0.010), Census 0.44 (p 0.053) | Little. 92 of its 118 calls are not A6·JNT calls; A6 does not test offsets |
| Does the **S1 → S2 → S3 profile** differ? | **A4·JNT**: coordinate-free step screen, joint test | Specific (50; 0, 0) with no coordinate; replicated (1.38 / 1.17, p 0.001) | Not in the list. A6·JNT agrees (Jaccard 0.75) and both collapse to 25 programs |
| Does the **positional profile** differ, **where**, and in **which direction**? | **A6·JNT**, the primary method: T_spatial, joint test | Specific (60; 2, 0); replicated (1.39 / 1.16, p 0.001); returns Δ(s), its peak and the direction there | Yes, as a reading rather than as calls: where the difference sits inside a segment (E2), and a direction that changes along the PT (E4) |
| The same question with a **gene-level calibrated list** | **A6·ORAc**: split-plot significant genes + ORA | The cleanest specificity (35; 0, 0) and the highest replication (1.53 / 1.35); same 25 programs | As A6·JNT |
| **Which genes** drive a call? | The program gene heatmaps of A6·JNT calls (section 8) | Shows driver genes, opposite-direction members and each specimen | Rows are positions, not averages |
| Is anything changing in this pathway **at all**? | A self-contained test (A6·SCs, or a score in the split plot, C·*·SP) | Specific (0 relabeled calls), but calls 73–100% of pathways and does not replicate (z ≈ 0.1), so it ranks nothing | The activity curves, as a picture only |
| Which genes share a **difference shape, whatever the library**? | **B2·split**: curve modules of split-plot genes; or B5 as a species-blind screen | B2 is specific (23; 0, 0) and replicated (1.19 / 1.02). B5 found one module that no pathway describes (section 8) | Module shapes are positional by construction |

**Methods to avoid for the species question.** Each was tested here and failed the relabeling rule,
or passed only by calling nothing or everything:
- GSEA on DESeq2 statistics (A1·GSEA, A3·GSEA: more relabeled than species calls);
- T_level and T_total (A2, A7);
- structure-level score GAMs (C·*·GAM) and the structure-level global test (A6·SCu): 12,000
  structures treated as replicates;
- ORA on the top 10% of T_spatial (A6·ORAr: 118; 41, 42);
- comparing per-group association lists (A8);
- curve modules from unselected or T_total-ranked genes (B1, B4);
- pseudobulk joint rank tests (A1·JNT, A3·JNT): no species calls.

Preranked GSEA on an unsigned positional statistic (A4·GSEA, A6·GSEA) passes the rule. It calls
about 300 pathways and replicates weakly (Census z 0.3, p 0.07–0.10).

**Gene-first against pathway-first.** Gene-first modules never beat pathway-first programs on
specificity or replication.
- The specific gene-first methods call 23, 84 and 3 pathways (B2, B3, B5).
- Only B2 replicates, at z 1.19 / 1.02, below A6·JNT and A6·ORAc.
- They recover 7, 7 and 3 of the primary method's 25 programs.

They add interpretability once. B5's module 1 is 46 co-expressed genes that rise into mouse S3 and
are mostly higher in mouse (Slc22a7, Slc5a8, Slc7a13, Cyp7b1, Cd36). Its curve differs between
species (q 4e-44, relabelings 0.93 and 0.99), but no library pathway is enriched in it, so a
pathway-first analysis cannot name it.

**The cost of running everything** is small. Every method here takes seconds to minutes of
serial-equivalent compute; the largest are B3 (180 s) and the per-group association (173 s). The
notebook's wall time, with the parallel tasks and the cache, is printed below.

This notebook takes SCF13 as exact. With the real coordinate, notebooks 57 and 58 show that
within-segment readings need the step nulls and replication checks before they are claimed.

## 10 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'n_structures': int(len(position)), 'n_genes': len(genes),
          'n_pathways': len(pathways), 'n_jobs': N_JOBS, 'methods': TABLE.reset_index().to_dict(orient='records'),
          'misses': misses.to_dict(), 'examples': {r: NAMES[p] for r, p in EXAMPLES.items()},
          'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Notebook wall time {record['wall_seconds'] / 60:.1f} min with {N_JOBS} workers")